# Experiment: MSFT 1-Minute Integrated Forecast Stack (v10)

1. Combines the strongest non-conflicting ideas from Phases 1 to 6 in `UPDATE.md`.
2. Uses technical indicators and regime features as inputs, then feeds them into a hybrid iTransformer + GRU + frequency encoder.
3. Adds retrieval-augmented inference, regime-aware rolling temperatures, and keeps the RL decision layer as a post-forecast section driven by strictly causal rolling predictions.

Integrated design:
- Phase 1: technical indicators
- Phase 2: regime detection in features and rolling inference
- Phase 3 + 4: hybrid time-domain, iTransformer, and frequency encoder
- Phase 5: retrieval-augmented path adjustment
- Phase 6: PPO-based decision layer over rolling outputs


## Package Installation & Imports

In [ ]:
import importlib.util
import subprocess
import sys

required = {
    'alpaca': 'alpaca-py',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'pandas_market_calendars': 'pandas-market-calendars',
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
else:
    print('All required third-party packages are already installed.')

In [ ]:
from __future__ import annotations
import copy
import math
import os
import random
import time
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from enum import Enum
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
import pandas_market_calendars as mcal
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Normal
from alpaca.data.enums import DataFeed
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from IPython.display import display
from matplotlib import pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.patches import Patch, Rectangle
from torch.utils.data import DataLoader, Dataset

print("Imports complete")


## Random Seed & Device Setup

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_TOTAL_MEM_GB = 0.0
if torch.cuda.is_available():
    GPU_TOTAL_MEM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
LOW_VRAM_MODE = bool(torch.cuda.is_available() and GPU_TOTAL_MEM_GB <= 10.0)
ROLLING_TRAIN_DEVICE = torch.device('cpu') if LOW_VRAM_MODE else DEVICE
RAG_DEVICE = torch.device('cpu') if LOW_VRAM_MODE else DEVICE

def cuda_cleanup() -> None:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'Using device: {DEVICE}')
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print({
    'low_vram_mode': LOW_VRAM_MODE,
    'gpu_total_mem_gb': round(GPU_TOTAL_MEM_GB, 2),
    'rolling_train_device': str(ROLLING_TRAIN_DEVICE),
    'rag_device': str(RAG_DEVICE),
})


## Regime Detection Components

In [ ]:
class MarketRegime(Enum):
    """Market regime classification."""
    NORMAL = "normal"
    ELEVATED = "elevated"
    CRISIS = "crisis"

    def __str__(self):
        return self.value

    @property
    def color(self) -> str:
        """Get visualization color for regime."""
        return {
            MarketRegime.NORMAL: '#00FF00',   # Green
            MarketRegime.ELEVATED: '#FFA500', # Orange
            MarketRegime.CRISIS: '#FF0000',   # Red
        }[self]

    @property
    def display_name(self) -> str:
        """Get display name for regime."""
        return {
            MarketRegime.NORMAL: 'NORMAL',
            MarketRegime.ELEVATED: 'ELEVATED',
            MarketRegime.CRISIS: 'CRISIS',
        }[self]


@dataclass
class RegimeConfig:
    """Configuration for regime thresholds and behavior."""
    # Turbulence thresholds (percentiles of historical distribution)
    normal_threshold: float = 0.75      # 75th percentile
    elevated_threshold: float = 0.90    # 90th percentile

    # Temperature adjustments (multiplied with base temperature)
    normal_temp_mult: float = 1.0
    elevated_temp_mult: float = 1.3
    crisis_temp_mult: float = 1.8

    # Position sizing adjustments (for trading)
    normal_position_mult: float = 1.0
    elevated_position_mult: float = 0.7
    crisis_position_mult: float = 0.3

    # Lookback for regime calculation
    lookback: int = 60


class TurbulenceIndexCalculator:
    """
    Calculates Kritzman-Li turbulence index using Mahalanobis distance.

    Turbulence measures the statistical unusualness of returns relative to
    recent history. High turbulence = market stress.

    Formula: d_t = (r_t - mu)^T * Sigma^{-1} * (r_t - mu)
    where r_t = return vector, mu = mean, Sigma = covariance
    """

    def __init__(self, lookback: int = 60):
        self.lookback = lookback
        self._history: list = []
        self._percentiles: Dict[float, float] = {}

    def update(self, returns: np.ndarray) -> float:
        """
        Calculate turbulence for current return vector.

        Args:
            returns: Vector of returns (can be single asset or multi-asset)

        Returns:
            Turbulence score (Mahalanobis distance)
        """
        self._history.append(returns)

        # Need enough history
        if len(self._history) < self.lookback:
            return 0.0

        # Keep only recent history
        self._history = self._history[-self.lookback:]

        # Calculate mean and covariance
        history_array = np.array(self._history)
        mean = np.mean(history_array, axis=0)

        # Regularized covariance (add small diagonal for stability)
        cov = np.cov(history_array.T)
        if cov.ndim < 2:
            cov = np.array([[cov]])
        cov += np.eye(cov.shape[0]) * 1e-6

        # Mahalanobis distance
        diff = returns - mean
        try:
            inv_cov = np.linalg.inv(cov)
            turbulence = np.sqrt(diff @ inv_cov @ diff)
        except np.linalg.LinAlgError:
            # Fallback to Euclidean distance if singular
            turbulence = np.linalg.norm(diff)

        return float(turbulence)

    def calibrate_percentiles(self, historical_turbulence: np.ndarray):
        """
        Calibrate percentile thresholds from historical data.

        Args:
            historical_turbulence: Array of historical turbulence values
        """
        clean = historical_turbulence[~np.isnan(historical_turbulence)]
        clean = clean[clean > 0]  # Remove zeros
        if len(clean) == 0:
            return
        self._percentiles = {
            0.50: np.percentile(clean, 50),
            0.75: np.percentile(clean, 75),
            0.90: np.percentile(clean, 90),
            0.95: np.percentile(clean, 95),
        }


class MarketRegimeDetector:
    """
    Detects market regime based on turbulence and volatility indicators.

    Integrates multiple signals:
    1. Turbulence index (statistical unusualness)
    2. ATR percentile (volatility level)
    """

    def __init__(self, config: RegimeConfig = None):
        self.config = config or RegimeConfig()
        self.turbulence_calc = TurbulenceIndexCalculator(self.config.lookback)
        self._atr_history: list = []
        self._turbulence_history: list = []
        self._current_regime: MarketRegime = MarketRegime.NORMAL
        self._regime_history: List[Tuple[pd.Timestamp, MarketRegime]] = []

    def detect_regime(self, returns: np.ndarray, atr: float, timestamp: pd.Timestamp = None) -> MarketRegime:
        """
        Detect current market regime.

        Args:
            returns: Current return vector
            atr: Current ATR value
            timestamp: Optional timestamp for logging

        Returns:
            MarketRegime classification
        """
        # Update turbulence
        turbulence = self.turbulence_calc.update(returns)

        # Update histories
        self._atr_history.append(atr)
        self._atr_history = self._atr_history[-self.config.lookback:]

        self._turbulence_history.append(turbulence)
        self._turbulence_history = self._turbulence_history[-self.config.lookback:]

        # Need enough history
        if len(self._atr_history) < self.config.lookback:
            regime = MarketRegime.NORMAL
        else:
            # Calculate percentiles
            atr_percentile = np.mean(np.array(self._atr_history) <= atr)

            # Get turbulence percentile
            turb_history = np.array([t for t in self._turbulence_history if t > 0])
            if len(turb_history) > 0:
                turb_percentile = np.mean(turb_history <= turbulence)
            else:
                turb_percentile = 0.5

            # Combined regime detection
            # Crisis: both high turbulence AND high volatility
            # Elevated: either high turbulence OR high volatility
            # Normal: neither

            if turb_percentile > self.config.elevated_threshold and atr_percentile > self.config.elevated_threshold:
                regime = MarketRegime.CRISIS
            elif turb_percentile > self.config.normal_threshold or atr_percentile > self.config.normal_threshold:
                regime = MarketRegime.ELEVATED
            else:
                regime = MarketRegime.NORMAL

        # Log regime transition
        if timestamp is not None and regime != self._current_regime:
            self._regime_history.append((timestamp, regime))

        self._current_regime = regime
        return regime

    def get_temperature_multiplier(self) -> float:
        """Get temperature adjustment for current regime."""
        multipliers = {
            MarketRegime.NORMAL: self.config.normal_temp_mult,
            MarketRegime.ELEVATED: self.config.elevated_temp_mult,
            MarketRegime.CRISIS: self.config.crisis_temp_mult
        }
        return multipliers.get(self._current_regime, 1.0)

    def get_position_multiplier(self) -> float:
        """Get position sizing adjustment for current regime."""
        multipliers = {
            MarketRegime.NORMAL: self.config.normal_position_mult,
            MarketRegime.ELEVATED: self.config.elevated_position_mult,
            MarketRegime.CRISIS: self.config.crisis_position_mult
        }
        return multipliers.get(self._current_regime, 1.0)

    def should_halt_trading(self) -> bool:
        """Check if trading should be halted (crisis regime)."""
        return self._current_regime == MarketRegime.CRISIS

    def get_regime_history(self) -> List[Tuple[pd.Timestamp, MarketRegime]]:
        """Get history of regime transitions."""
        return self._regime_history.copy()

    def get_current_regime(self) -> MarketRegime:
        """Get current regime."""
        return self._current_regime


def calculate_historical_turbulence(df: pd.DataFrame, lookback: int = 60) -> pd.Series:
    """
    Calculate historical turbulence for entire DataFrame.

    Args:
        df: DataFrame with returns column
        lookback: Lookback window for turbulence calculation

    Returns:
        Series of turbulence values
    """
    calculator = TurbulenceIndexCalculator(lookback)
    turbulence = []

    for i in range(len(df)):
        if i < lookback or 'returns' not in df.columns:
            turbulence.append(0.0)
        else:
            ret = df['returns'].iloc[i]
            if pd.isna(ret):
                turbulence.append(0.0)
            else:
                turb = calculator.update(np.array([ret]))
                turbulence.append(turb)

    return pd.Series(turbulence, index=df.index)

## Configuration

In [ ]:
# Data Configuration
SYMBOL = 'MSFT'
LOOKBACK_DAYS = 120
OHLC_COLS = ['Open', 'High', 'Low', 'Close']
RAW_COLS = OHLC_COLS + ['Volume', 'TradeCount', 'VWAP']
BASE_CORE_FEATURES = [
    'rOpen',
    'rHigh',
    'rLow',
    'rClose',
    'logVolChange',
    'logTradeCountChange',
    'vwapDelta',
    'rangeFrac',
    'orderFlowProxy',
    'tickPressure',
]
TECHNICAL_FEATURE_COLS = [
    'sma_5',
    'sma_10',
    'sma_20',
    'sma_50',
    'ema_12',
    'ema_26',
    'macd_line',
    'macd_signal',
    'macd_histogram',
    'macd_momentum',
    'rsi_14',
    'rsi_14_slope',
    'stoch_k',
    'stoch_d',
    'bb_upper',
    'bb_lower',
    'bb_width',
    'bb_position',
    'atr_14',
    'atr_14_pct',
    'obv',
    'obv_slope',
    'vwap_20',
    'vwap_20_dev',
    'price_momentum_5',
    'price_momentum_10',
    'price_momentum_20',
    'body_size',
    'body_pct',
    'upper_shadow',
    'lower_shadow',
    'direction',
]
REGIME_FEATURE_COLS = [
    'returns',
    'turbulence_60',
    'regime_indicator',
]
V10_FEATURE_COLS = [
    'sma_5',
    'sma_10',
    'sma_20',
    'sma_50',
    'ema_12',
    'ema_26',
    'macd_line',
    'macd_signal',
    'macd_histogram',
    'macd_momentum',
    'rsi_14',
    'rsi_14_slope',
    'stoch_k',
    'stoch_d',
    'bb_upper',
    'bb_lower',
    'bb_width',
    'bb_position',
    'atr_14',
    'atr_14_pct',
    'obv',
    'obv_slope',
    'vwap_20',
    'vwap_20_dev',
    'price_momentum_5',
    'price_momentum_10',
    'price_momentum_20',
    'body_size',
    'body_pct',
    'upper_shadow',
    'lower_shadow',
    'direction',
    'returns',
    'turbulence_60',
    'regime_indicator',
]
BASE_FEATURE_COLS = BASE_CORE_FEATURES + V10_FEATURE_COLS
TARGET_COLS = ['rOpen', 'rHigh', 'rLow', 'rClose']
INPUT_EXTRA_COL = 'imputedFracWindow'

HORIZON = 50
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
LOOKBACK_CANDIDATES = [64, 96, 160, 256]
DEFAULT_LOOKBACK = 96
ENABLE_LOOKBACK_SWEEP = True
SKIP_OPEN_BARS_TARGET = 6


In [ ]:
# Model Configuration
HIDDEN_SIZE = 256  # Increased for better generation capacity
NUM_LAYERS = 2
DROPOUT = 0.20     # Slightly higher for stochasticity
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-5
BASE_BATCH_SIZE = 256
BATCH_SIZE = 32 if LOW_VRAM_MODE else BASE_BATCH_SIZE
ROLLING_TRAIN_BATCH_SIZE = 16 if LOW_VRAM_MODE else min(128, BASE_BATCH_SIZE)


In [ ]:
# Integrated architecture configuration
D_MODEL = 128
N_HEADS = 8
N_LAYERS = 2
USE_FREQUENCY = True
FREQ_N_FFT = 16
FREQ_HOP_LENGTH = 4
FREQ_OUT_CHANNELS = 64
MULTISCALE_N_FFTS = [8, 16, 32]


In [ ]:
# Training Configuration
SWEEP_MAX_EPOCHS = 15
SWEEP_PATIENCE = 5
FINAL_MAX_EPOCHS = 60  # More epochs for convergence
FINAL_PATIENCE = 12
TF_START = 1.0
TF_END = 0.0
TF_DECAY_RATE = 0.95


In [ ]:
# Loss Configuration
RANGE_LOSS_WEIGHT = 0.3
VOLATILITY_WEIGHT = 0.5  # Encourage proper volatility
DIR_PENALTY_WEIGHT = 0.1
STEP_LOSS_POWER = 1.5

In [ ]:
# Inference Configuration - tuned for realistic 1-minute bars
SAMPLING_TEMPERATURE = 0.50
BASE_ENSEMBLE_SIZE = 16
ENSEMBLE_SIZE = 8 if LOW_VRAM_MODE else BASE_ENSEMBLE_SIZE
TREND_LOOKBACK_BARS = 20
STRONG_TREND_THRESHOLD = 0.002
VOLATILITY_SCALING = True
MIN_PREDICTED_VOL = 0.0001
LOG_SIGMA_MIN = math.log(MIN_PREDICTED_VOL)
LOG_SIGMA_MAX = math.log(0.01)


In [ ]:
# Phase 2: Regime Detection Configuration
REGIME_CONFIG = RegimeConfig(
    normal_threshold=0.75,
    elevated_threshold=0.90,
    normal_temp_mult=1.0,
    elevated_temp_mult=1.3,
    crisis_temp_mult=1.8,
    normal_position_mult=1.0,
    elevated_position_mult=0.7,
    crisis_position_mult=0.3,
    lookback=60,
)


In [ ]:
# Phase 5: Retrieval-Augmented Pattern Memory
RAG_EMBEDDING_DIM = 64
RAG_K_RETRIEVE = 5
RAG_BLEND_WEIGHT = 0.25
RAG_MAX_PATTERNS = 3000 if LOW_VRAM_MODE else 4000
ROLLING_RAG_MAX_PATTERNS = 1500 if LOW_VRAM_MODE else RAG_MAX_PATTERNS
RAG_ENCODER_HIDDEN = 128
RAG_ENCODER_LAYERS = 2
RAG_BUILD_BATCH_SIZE = 32 if LOW_VRAM_MODE else 128


In [ ]:
# Phase 6: RL Decision Layer Configuration
RUN_RL_TRAINING = False
RL_CONFIG = {
    'initial_balance': 100000.0,
    'transaction_cost': 0.001,
    'max_position': 1.0,
    'dsr_eta': 0.1,
    'ppo_lr': 3e-4,
    'ppo_gamma': 0.99,
    'ppo_gae_lambda': 0.95,
    'ppo_clip_epsilon': 0.2,
    'ppo_value_coef': 0.5,
    'ppo_entropy_coef': 0.01,
    'ppo_max_grad_norm': 0.5,
    'ppo_rollout_length': 512,
    'ppo_update_epochs': 4,
    'ppo_batch_size': 64,
    'rl_training_steps': 5000,
}


In [ ]:
# Data Processing Configuration
STANDARDIZE_TARGETS = False
APPLY_CLIPPING = True
CLIP_QUANTILES = (0.001, 0.999)
DIRECTION_EPS = 0.0001
STD_RATIO_TARGET_MIN = 0.3

In [ ]:
# Alpaca API Configuration
ALPACA_FEED = os.getenv('ALPACA_FEED', 'iex').strip().lower()
SESSION_TZ = 'America/New_York'
REQUEST_CHUNK_DAYS = 5
MAX_REQUESTS_PER_MINUTE = 120
MAX_RETRIES = 5
MAX_SESSION_FILL_RATIO = 0.15

In [ ]:
# Print Configuration Summary
summary = {
    'version': 'v10',
    'phase': 'Integrated Phase Stack',
    'symbol': SYMBOL,
    'lookback_days': LOOKBACK_DAYS,
    'horizon': HORIZON,
    'feature_dim': len(BASE_FEATURE_COLS),
    'ensemble_size': ENSEMBLE_SIZE,
    'sampling_temperature': SAMPLING_TEMPERATURE,
    'device': str(DEVICE),
}

if 'TECHNICAL_FEATURE_COLS' in globals():
    summary['technical_feature_count'] = len(TECHNICAL_FEATURE_COLS)
if 'REGIME_FEATURE_COLS' in globals():
    summary['regime_feature_count'] = len(REGIME_FEATURE_COLS)
if 'D_MODEL' in globals():
    summary['d_model'] = D_MODEL
if 'N_HEADS' in globals():
    summary['n_heads'] = N_HEADS
if 'N_LAYERS' in globals():
    summary['n_layers'] = N_LAYERS
if 'USE_FREQUENCY' in globals():
    summary['use_frequency'] = USE_FREQUENCY
    summary['multiscale_n_ffts'] = MULTISCALE_N_FFTS
if 'RAG_EMBEDDING_DIM' in globals():
    summary['rag_embedding_dim'] = RAG_EMBEDDING_DIM
    summary['rag_k_retrieve'] = RAG_K_RETRIEVE
    summary['rag_blend_weight'] = RAG_BLEND_WEIGHT
if 'RL_CONFIG' in globals():
    summary['rl_training_steps'] = RL_CONFIG['rl_training_steps']
    summary['run_rl_training'] = RUN_RL_TRAINING

print(summary)


## Data Fetching Functions

In [ ]:
class RequestPacer:
    def __init__(self, max_calls_per_minute: int):
        if max_calls_per_minute <= 0:
            raise ValueError('max_calls_per_minute must be >0')
        self.min_interval = 60.0 / float(max_calls_per_minute)
        self.last_call_ts = 0.0

    def wait(self) -> None:
        now = time.monotonic()
        elapsed = now - self.last_call_ts
        if elapsed < self.min_interval:
            time.sleep(self.min_interval - elapsed)
        self.last_call_ts = time.monotonic()

In [ ]:
def _require_alpaca_credentials() -> tuple[str, str]:
    api_key = os.getenv('ALPACA_API_KEY')
    secret_key = os.getenv('ALPACA_SECRET_KEY')
    if not api_key or not secret_key:
        raise RuntimeError('Missing ALPACA_API_KEY / ALPACA_SECRET_KEY.')
    return api_key, secret_key

def _resolve_feed(feed_name: str) -> DataFeed:
    mapping = {'iex': DataFeed.IEX, 'sip': DataFeed.SIP, 'delayed_sip': DataFeed.DELAYED_SIP}
    k = feed_name.strip().lower()
    if k not in mapping:
        raise ValueError(f'Unsupported ALPACA_FEED={feed_name!r}. Use one of: {list(mapping)}')
    return mapping[k]

In [ ]:
def fetch_bars_alpaca(symbol: str, lookback_days: int) -> tuple[pd.DataFrame, int]:
    api_key, secret_key = _require_alpaca_credentials()
    client = StockHistoricalDataClient(api_key=api_key, secret_key=secret_key)
    feed = _resolve_feed(ALPACA_FEED)
    pacer = RequestPacer(MAX_REQUESTS_PER_MINUTE)

    end_ts = datetime.now(timezone.utc).replace(second=0, microsecond=0)
    if ALPACA_FEED in {'sip', 'delayed_sip'}:
        end_ts = end_ts - timedelta(minutes=20)
    start_ts = end_ts - timedelta(days=lookback_days)

    parts = []
    cursor = start_ts
    calls = 0

    while cursor < end_ts:
        chunk_end = min(cursor + timedelta(days=REQUEST_CHUNK_DAYS), end_ts)
        chunk = None
        for attempt in range(1, MAX_RETRIES + 1):
            pacer.wait()
            calls += 1
            try:
                req = StockBarsRequest(
                    symbol_or_symbols=[symbol],
                    timeframe=TimeFrame.Minute,
                    start=cursor,
                    end=chunk_end,
                    feed=feed,
                    limit=10000,
                )
                chunk = client.get_stock_bars(req).df
                break
            except Exception as exc:
                msg = str(exc).lower()
                if ('429' in msg or 'rate limit' in msg) and attempt < MAX_RETRIES:
                    backoff = min(2 ** attempt, 30)
                    print(f'Rate-limited; sleeping {backoff}s (attempt {attempt}/{MAX_RETRIES}).')
                    time.sleep(backoff)
                    continue
                if ('subscription' in msg or 'forbidden' in msg) and ALPACA_FEED != 'iex':
                    raise RuntimeError('Feed unavailable for account. Use ALPACA_FEED=iex or upgrade subscription.') from exc
                raise
        if chunk is not None and not chunk.empty:
            d = chunk.reset_index().rename(columns={
                'timestamp': 'Datetime', 'open': 'Open', 'high': 'High',
                'low': 'Low', 'close': 'Close', 'volume': 'Volume',
                'trade_count': 'TradeCount', 'vwap': 'VWAP',
            })
            if 'Volume' not in d.columns:
                d['Volume'] = 0.0
            if 'TradeCount' not in d.columns:
                d['TradeCount'] = 0.0
            if 'VWAP' not in d.columns:
                d['VWAP'] = d['Close']

            need = ['Datetime'] + RAW_COLS
            d['Datetime'] = pd.to_datetime(d['Datetime'], utc=True)
            d = d[need].dropna(subset=OHLC_COLS).set_index('Datetime').sort_index()
            parts.append(d)
        cursor = chunk_end

    if not parts:
        raise RuntimeError('No bars returned from Alpaca.')
    out = pd.concat(parts, axis=0).sort_index()
    out = out[~out.index.duplicated(keep='last')]
    return out.astype(np.float32), calls

In [ ]:
def sessionize_with_calendar(df_utc: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    if df_utc.empty:
        raise RuntimeError('Input bars are empty.')

    idx = pd.DatetimeIndex(df_utc.index)
    if idx.tz is None:
        idx = idx.tz_localize('UTC')
    else:
        idx = idx.tz_convert('UTC')

    df_utc = df_utc.copy()
    df_utc.index = idx

    cal = mcal.get_calendar('XNYS')
    sched = cal.schedule(
        start_date=(idx.min() - pd.Timedelta(days=2)).date(),
        end_date=(idx.max() + pd.Timedelta(days=2)).date(),
    )

    pieces = []
    fill_ratios = []

    for sid, (_, row) in enumerate(sched.iterrows()):
        open_ts = pd.Timestamp(row['market_open'])
        close_ts = pd.Timestamp(row['market_close'])

        if open_ts.tzinfo is None:
            open_ts = open_ts.tz_localize('UTC')
        else:
            open_ts = open_ts.tz_convert('UTC')
        if close_ts.tzinfo is None:
            close_ts = close_ts.tz_localize('UTC')
        else:
            close_ts = close_ts.tz_convert('UTC')

        exp_idx = pd.date_range(open_ts, close_ts, freq='1min', inclusive='left')
        if len(exp_idx) == 0:
            continue

        day = df_utc[(df_utc.index >= open_ts) & (df_utc.index < close_ts)]
        day = day.reindex(exp_idx)
        imputed = day[OHLC_COLS].isna().any(axis=1).to_numpy()
        fill_ratio = float(imputed.mean())

        if fill_ratio >= 1.0 or fill_ratio > MAX_SESSION_FILL_RATIO:
            continue

        day[OHLC_COLS + ['VWAP']] = day[OHLC_COLS + ['VWAP']].ffill().bfill()
        if day['VWAP'].isna().all():
            day['VWAP'] = day['Close']
        else:
            day['VWAP'] = day['VWAP'].fillna(day['Close'])

        day['Volume'] = day['Volume'].fillna(0.0)
        day['TradeCount'] = day['TradeCount'].fillna(0.0)
        day['is_imputed'] = imputed.astype(np.int8)
        day['session_id'] = int(sid)
        day['bar_in_session'] = np.arange(len(day), dtype=np.int32)
        day['session_len'] = int(len(day))

        if day[RAW_COLS].isna().any().any():
            raise RuntimeError('NaNs remain after per-session fill.')
        pieces.append(day)
        fill_ratios.append(fill_ratio)

    if not pieces:
        raise RuntimeError('No sessions kept after calendar filtering.')

    out = pd.concat(pieces, axis=0).sort_index()
    out.index = out.index.tz_convert(SESSION_TZ).tz_localize(None)
    out = out.copy()

    for c in RAW_COLS:
        out[c] = out[c].astype(np.float32)
    out['is_imputed'] = out['is_imputed'].astype(np.int8)
    out['session_id'] = out['session_id'].astype(np.int32)
    out['bar_in_session'] = out['bar_in_session'].astype(np.int32)
    out['session_len'] = out['session_len'].astype(np.int32)

    meta = {
        'calendar_sessions_total': int(len(sched)),
        'kept_sessions': int(len(pieces)),
        'avg_fill_ratio_kept': float(np.mean(fill_ratios)) if fill_ratios else float('nan'),
    }
    return out, meta

## Fetch Data from Alpaca

In [ ]:
raw_df_utc, api_calls = fetch_bars_alpaca(SYMBOL, LOOKBACK_DAYS)
price_df, session_meta = sessionize_with_calendar(raw_df_utc)
print(f'Raw rows from Alpaca: {len(raw_df_utc):,}')
print(f'Sessionized rows kept: {len(price_df):,}')
print('Session meta:', session_meta)

min_needed = max(LOOKBACK_CANDIDATES) + HORIZON + 1000
if len(price_df) < min_needed:
    raise RuntimeError(f'Not enough rows after session filtering ({len(price_df)}). Need at least {min_needed}.')

## Feature Engineering Functions (Integrated Phase 1 + 2)

### Technical Indicator Engine

In [ ]:
class TechnicalIndicatorCalculator:
    """Compute technical indicators used in Phase 1."""

    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()
        required = {'Open', 'High', 'Low', 'Close', 'Volume'}
        missing = required.difference(self.df.columns)
        if missing:
            raise ValueError(f'Missing columns for indicators: {sorted(missing)}')

    def add_sma(self, periods: List[int] = [5, 10, 20, 50]) -> 'TechnicalIndicatorCalculator':
        for period in periods:
            self.df[f'sma_{period}'] = self.df['Close'].rolling(period).mean()
        return self

    def add_ema(self, periods: List[int] = [12, 26]) -> 'TechnicalIndicatorCalculator':
        for period in periods:
            self.df[f'ema_{period}'] = self.df['Close'].ewm(span=period, adjust=False).mean()
        return self

    def add_macd(self, fast: int = 12, slow: int = 26, signal: int = 9) -> 'TechnicalIndicatorCalculator':
        ema_fast = self.df['Close'].ewm(span=fast, adjust=False).mean()
        ema_slow = self.df['Close'].ewm(span=slow, adjust=False).mean()
        self.df['macd_line'] = ema_fast - ema_slow
        self.df['macd_signal'] = self.df['macd_line'].ewm(span=signal, adjust=False).mean()
        self.df['macd_histogram'] = self.df['macd_line'] - self.df['macd_signal']
        self.df['macd_momentum'] = self.df['macd_histogram'].diff()
        return self

    def add_rsi(self, period: int = 14) -> 'TechnicalIndicatorCalculator':
        delta = self.df['Close'].diff()
        gain = delta.clip(lower=0.0)
        loss = (-delta.clip(upper=0.0))
        avg_gain = gain.ewm(alpha=1 / period, adjust=False).mean()
        avg_loss = loss.ewm(alpha=1 / period, adjust=False).mean()
        rs = avg_gain / avg_loss.replace(0.0, np.nan)
        self.df[f'rsi_{period}'] = 100.0 - (100.0 / (1.0 + rs))
        self.df[f'rsi_{period}_slope'] = self.df[f'rsi_{period}'].diff(5)
        return self

    def add_stochastic(self, k_period: int = 14, d_period: int = 3) -> 'TechnicalIndicatorCalculator':
        low_min = self.df['Low'].rolling(k_period).min()
        high_max = self.df['High'].rolling(k_period).max()
        denom = (high_max - low_min).replace(0.0, np.nan)
        self.df['stoch_k'] = 100.0 * (self.df['Close'] - low_min) / denom
        self.df['stoch_d'] = self.df['stoch_k'].rolling(d_period).mean()
        return self

    def add_bollinger_bands(self, period: int = 20, std_dev: float = 2.0) -> 'TechnicalIndicatorCalculator':
        sma = self.df['Close'].rolling(period).mean()
        std = self.df['Close'].rolling(period).std()
        self.df['bb_upper'] = sma + std_dev * std
        self.df['bb_lower'] = sma - std_dev * std
        width = (self.df['bb_upper'] - self.df['bb_lower']).replace(0.0, np.nan)
        self.df['bb_width'] = width / sma.replace(0.0, np.nan)
        self.df['bb_position'] = (self.df['Close'] - self.df['bb_lower']) / width
        return self

    def add_atr(self, period: int = 14) -> 'TechnicalIndicatorCalculator':
        high_low = self.df['High'] - self.df['Low']
        high_close = (self.df['High'] - self.df['Close'].shift()).abs()
        low_close = (self.df['Low'] - self.df['Close'].shift()).abs()
        true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
        self.df['atr_14'] = true_range.ewm(alpha=1 / period, adjust=False).mean()
        self.df['atr_14_pct'] = self.df['atr_14'] / self.df['Close'].replace(0.0, np.nan)
        return self

    def add_obv(self) -> 'TechnicalIndicatorCalculator':
        delta = np.sign(self.df['Close'].diff().fillna(0.0))
        self.df['obv'] = (delta * self.df['Volume']).cumsum()
        self.df['obv_slope'] = self.df['obv'].diff(5)
        return self

    def add_vwap(self, period: int = 20) -> 'TechnicalIndicatorCalculator':
        typical_price = (self.df['High'] + self.df['Low'] + self.df['Close']) / 3.0
        tpv = typical_price * self.df['Volume']
        rolling_volume = self.df['Volume'].rolling(period).sum().replace(0.0, np.nan)
        self.df['vwap_20'] = tpv.rolling(period).sum() / rolling_volume
        self.df['vwap_20_dev'] = (self.df['Close'] - self.df['vwap_20']) / self.df['vwap_20'].replace(0.0, np.nan)
        return self

    def add_price_momentum(self, periods: List[int] = [5, 10, 20]) -> 'TechnicalIndicatorCalculator':
        for period in periods:
            self.df[f'price_momentum_{period}'] = self.df['Close'] / self.df['Close'].shift(period) - 1.0
        return self

    def add_candle_features(self) -> 'TechnicalIndicatorCalculator':
        spread = (self.df['High'] - self.df['Low']).replace(0.0, np.nan)
        self.df['body_size'] = (self.df['Close'] - self.df['Open']).abs()
        self.df['body_pct'] = self.df['body_size'] / spread
        self.df['upper_shadow'] = self.df['High'] - self.df[['Open', 'Close']].max(axis=1)
        self.df['lower_shadow'] = self.df[['Open', 'Close']].min(axis=1) - self.df['Low']
        self.df['direction'] = np.where(self.df['Close'] > self.df['Open'], 1.0, np.where(self.df['Close'] < self.df['Open'], -1.0, 0.0))
        return self

    def get_all_indicators(self) -> pd.DataFrame:
        return (
            self.add_sma()
            .add_ema()
            .add_macd()
            .add_rsi()
            .add_stochastic()
            .add_bollinger_bands()
            .add_atr()
            .add_obv()
            .add_vwap()
            .add_price_momentum()
            .add_candle_features()
            .df
        )


def calculate_technical_features(df: pd.DataFrame) -> pd.DataFrame:
    return TechnicalIndicatorCalculator(df).get_all_indicators()


In [ ]:
def enforce_candle_validity(ohlc: np.ndarray) -> np.ndarray:
    out = np.asarray(ohlc, dtype=np.float32)
    o, h, l, c = out[:, 0], out[:, 1], out[:, 2], out[:, 3]
    out[:, 1] = np.maximum.reduce([h, o, c])
    out[:, 2] = np.minimum.reduce([l, o, c])
    return out

def returns_to_prices_seq(return_ohlc: np.ndarray, last_close: float) -> np.ndarray:
    seq = []
    prev_close = float(last_close)
    for rO, rH, rL, rC in np.asarray(return_ohlc, dtype=np.float32):
        o = prev_close * np.exp(float(rO))
        h = prev_close * np.exp(float(rH))
        l = prev_close * np.exp(float(rL))
        c = prev_close * np.exp(float(rC))
        cand = enforce_candle_validity(np.array([[o, h, l, c]], dtype=np.float32))[0]
        seq.append(cand)
        prev_close = float(cand[3])
    return np.asarray(seq, dtype=np.float32)

In [ ]:
def encode_regime_state(turbulence: pd.Series, atr_pct: pd.Series, lookback: int) -> pd.Series:
    labels = np.ones(len(turbulence), dtype=np.float32)
    turb_values = turbulence.fillna(0.0).to_numpy(np.float32)
    atr_values = atr_pct.fillna(0.0).to_numpy(np.float32)

    for i in range(len(labels)):
        start = max(0, i - lookback + 1)
        turb_window = turb_values[start : i + 1]
        atr_window = atr_values[start : i + 1]
        if len(turb_window) < lookback:
            labels[i] = 1.0
            continue
        turb_pct = float(np.mean(turb_window <= turb_window[-1]))
        atr_pct_rank = float(np.mean(atr_window <= atr_window[-1]))
        if turb_pct > REGIME_CONFIG.elevated_threshold and atr_pct_rank > REGIME_CONFIG.elevated_threshold:
            labels[i] = -1.0
        elif turb_pct > REGIME_CONFIG.normal_threshold or atr_pct_rank > REGIME_CONFIG.normal_threshold:
            labels[i] = 0.0
        else:
            labels[i] = 1.0
    return pd.Series(labels, index=turbulence.index, dtype=np.float32)


def build_feature_frame(df: pd.DataFrame) -> pd.DataFrame:
    eps = 1e-9
    g = df.groupby('session_id', sort=False)
    prev_close = g['Close'].shift(1).fillna(df['Open'])
    prev_vol = g['Volume'].shift(1).fillna(df['Volume'])
    prev_tc = g['TradeCount'].shift(1).fillna(df['TradeCount'])
    prev_imp = g['is_imputed'].shift(1).fillna(0).astype(bool)

    row_imputed = (df['is_imputed'].astype(bool) | prev_imp)
    row_open_skip = df['bar_in_session'].astype(int) < SKIP_OPEN_BARS_TARGET

    out = pd.DataFrame(index=df.index, dtype=np.float32)
    out['rOpen'] = np.log(df['Open'] / (prev_close + eps))
    out['rHigh'] = np.log(df['High'] / (prev_close + eps))
    out['rLow'] = np.log(df['Low'] / (prev_close + eps))
    out['rClose'] = np.log(df['Close'] / (prev_close + eps))
    out['logVolChange'] = np.log((df['Volume'] + 1.0) / (prev_vol + 1.0))
    out['logTradeCountChange'] = np.log((df['TradeCount'] + 1.0) / (prev_tc + 1.0))
    out['vwapDelta'] = np.log((df['VWAP'] + eps) / (df['Close'] + eps))
    out['rangeFrac'] = np.maximum(out['rHigh'] - out['rLow'], 0.0) / (np.abs(out['rClose']) + eps)

    signed_body = (df['Close'] - df['Open']) / ((df['High'] - df['Low']) + eps)
    out['orderFlowProxy'] = signed_body * np.log1p(df['Volume'])
    out['tickPressure'] = np.sign(df['Close'] - df['Open']) * np.log1p(df['TradeCount'])

    technical = calculate_technical_features(df[OHLC_COLS + ['Volume']].copy())
    for col in TECHNICAL_FEATURE_COLS:
        out[col] = technical[col].astype(np.float32)

    out['returns'] = out['rClose']
    out['turbulence_60'] = calculate_historical_turbulence(
        pd.DataFrame({'returns': out['returns']}, index=df.index),
        lookback=REGIME_CONFIG.lookback,
    )
    out['regime_indicator'] = encode_regime_state(
        turbulence=out['turbulence_60'],
        atr_pct=out['atr_14_pct'],
        lookback=REGIME_CONFIG.lookback,
    )

    out['row_imputed'] = row_imputed.astype(np.int8).to_numpy()
    out['row_open_skip'] = row_open_skip.astype(np.int8).to_numpy()
    out['prev_close'] = prev_close.astype(np.float32).to_numpy()

    out = out.replace([np.inf, -np.inf], np.nan).ffill().fillna(0.0)
    return out.astype(np.float32)


def build_target_frame(feat_df: pd.DataFrame) -> pd.DataFrame:
    return feat_df[TARGET_COLS].copy().astype(np.float32)


In [ ]:
feat_df = build_feature_frame(price_df)
target_df = build_target_frame(feat_df)

missing_feature_cols = [col for col in BASE_FEATURE_COLS if col not in feat_df.columns]
if missing_feature_cols:
    raise RuntimeError(f'Missing engineered features: {missing_feature_cols}')

print('Feature rows:', len(feat_df))
print('Feature dimension:', len(BASE_FEATURE_COLS))
print('Target columns:', list(target_df.columns))
print('Feature preview:', BASE_FEATURE_COLS[:5], '...', BASE_FEATURE_COLS[-5:])


## Windowing & Dataset Functions

In [ ]:
def split_points(n_rows: int) -> tuple[int, int]:
    tr = int(n_rows * TRAIN_RATIO)
    va = int(n_rows * (TRAIN_RATIO + VAL_RATIO))
    return tr, va

def build_walkforward_slices(price_df_full: pd.DataFrame) -> list[tuple[str, int, int]]:
    n = len(price_df_full)
    span = int(round(n * 0.85))
    shift = max(1, n - span)
    cands = [('slice_1', 0, min(span, n)), ('slice_2', shift, min(shift + span, n))]
    out = []
    seen = set()
    for name, a, b in cands:
        key = (a, b)
        if key in seen or b - a < max(LOOKBACK_CANDIDATES) + HORIZON + 1400:
            continue
        out.append((name, a, b))
        seen.add(key)
    return out if out else [('full', 0, n)]

In [ ]:
def make_multistep_windows(input_scaled, target_scaled, target_raw, row_imputed, row_open_skip, 
                           starts_prev_close, window, horizon):
    X, y_s, y_r, starts, prev_close = [], [], [], [], []
    dropped_target_imputed, dropped_target_open_skip = 0, 0
    n = len(input_scaled)

    for i in range(window, n - horizon + 1):
        if row_imputed[i:i+horizon].any():
            dropped_target_imputed += 1
            continue
        if row_open_skip[i:i+horizon].any():
            dropped_target_open_skip += 1
            continue

        xb = input_scaled[i-window:i]
        imp_frac = float(row_imputed[i-window:i].mean())
        imp_col = np.full((window, 1), imp_frac, dtype=np.float32)
        xb_aug = np.concatenate([xb, imp_col], axis=1)

        X.append(xb_aug)
        y_s.append(target_scaled[i:i+horizon])
        y_r.append(target_raw[i:i+horizon])
        starts.append(i)
        prev_close.append(starts_prev_close[i])

    return (np.asarray(X, dtype=np.float32), np.asarray(y_s, dtype=np.float32),
            np.asarray(y_r, dtype=np.float32), np.asarray(starts, dtype=np.int64),
            np.asarray(prev_close, dtype=np.float32), dropped_target_imputed, dropped_target_open_skip)

In [ ]:
class MultiStepDataset(Dataset):
    def __init__(self, X, y_s, y_r):
        self.X = torch.from_numpy(X).float()
        self.y_s = torch.from_numpy(y_s).float()
        self.y_r = torch.from_numpy(y_r).float()
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y_s[idx], self.y_r[idx]

In [ ]:
slices = build_walkforward_slices(price_df)
print('Walk-forward slices:', slices)

## Retrieval-Augmented Pattern Memory

In [ ]:
try:
    import faiss  # type: ignore
except Exception:
    faiss = None


@dataclass
class RetrievedPattern:
    sequence: torch.Tensor
    future_path: torch.Tensor
    similarity: float
    index: int


@dataclass
class PatternMatch:
    patterns: List[RetrievedPattern]
    avg_similarity: float = 0.0
    max_similarity: float = 0.0
    min_similarity: float = 0.0


class PatternEncoder(nn.Module):
    def __init__(self, input_size: int, hidden_size: int = 128, embedding_dim: int = 64, num_layers: int = 2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2 if num_layers > 1 else 0.0,
        )
        self.projection = nn.Sequential(
            nn.Linear(hidden_size, embedding_dim),
            nn.LayerNorm(embedding_dim),
            nn.Tanh(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, (hidden, _) = self.lstm(x)
        return self.projection(hidden[-1])


class PatternDatabase:
    def __init__(self, embedding_dim: int = 64, use_faiss: bool = True):
        self.embedding_dim = embedding_dim
        self.use_faiss = bool(use_faiss and faiss is not None)
        self.index = faiss.IndexFlatIP(embedding_dim) if self.use_faiss else None
        self.embeddings: Optional[torch.Tensor] = None
        self.sequences: List[torch.Tensor] = []
        self.future_paths: List[torch.Tensor] = []

    def add_patterns(self, embeddings: torch.Tensor, sequences: torch.Tensor, future_paths: torch.Tensor) -> None:
        embeddings = F.normalize(embeddings.detach().cpu().float(), p=2, dim=1)
        sequences = sequences.detach().cpu().float()
        future_paths = future_paths.detach().cpu().float()
        if self.use_faiss and self.index is not None:
            self.index.add(embeddings.numpy().astype('float32'))
        else:
            self.embeddings = embeddings if self.embeddings is None else torch.cat([self.embeddings, embeddings], dim=0)
        self.sequences.extend(list(sequences))
        self.future_paths.extend(list(future_paths))

    def size(self) -> int:
        if self.use_faiss and self.index is not None:
            return int(self.index.ntotal)
        return 0 if self.embeddings is None else int(self.embeddings.shape[0])

    def search(self, query_embedding: torch.Tensor, k: int = 5) -> List[RetrievedPattern]:
        if self.size() == 0:
            return []
        query = F.normalize(query_embedding.detach().cpu().view(1, -1).float(), p=2, dim=1)
        if self.use_faiss and self.index is not None:
            sims, idxs = self.index.search(query.numpy().astype('float32'), min(k, self.size()))
            scored = zip(sims[0].tolist(), idxs[0].tolist())
        else:
            assert self.embeddings is not None
            sims = torch.matmul(self.embeddings, query.squeeze(0))
            topk = torch.topk(sims, k=min(k, sims.numel()))
            scored = zip(topk.values.tolist(), topk.indices.tolist())
        results = []
        for sim, idx in scored:
            if idx < 0:
                continue
            results.append(
                RetrievedPattern(
                    sequence=self.sequences[idx],
                    future_path=self.future_paths[idx],
                    similarity=float(sim),
                    index=int(idx),
                )
            )
        return results


class RAGPatternRetriever(nn.Module):
    def __init__(self, input_size: int, embedding_dim: int = 64, k_retrieve: int = 5, hidden_size: int = 128, num_layers: int = 2):
        super().__init__()
        self.input_size = input_size
        self.embedding_dim = embedding_dim
        self.k_retrieve = k_retrieve
        self.encoder = PatternEncoder(
            input_size=input_size,
            hidden_size=hidden_size,
            embedding_dim=embedding_dim,
            num_layers=num_layers,
        )
        self.database: Optional[PatternDatabase] = None

    def ready(self) -> bool:
        return self.database is not None and self.database.size() > 0

    def build_database(self, sequences: torch.Tensor, future_paths: torch.Tensor, batch_size: int = RAG_BUILD_BATCH_SIZE) -> None:
        if len(sequences) == 0:
            self.database = PatternDatabase(self.embedding_dim, use_faiss=False)
            return
        self.database = PatternDatabase(self.embedding_dim, use_faiss=False)
        device = next(self.encoder.parameters()).device
        embeddings: List[torch.Tensor] = []
        self.encoder.eval()
        with torch.no_grad():
            for start in range(0, len(sequences), batch_size):
                batch = sequences[start : start + batch_size].to(device)
                embeddings.append(self.encoder(batch).cpu())
        self.database.add_patterns(
            embeddings=torch.cat(embeddings, dim=0),
            sequences=sequences.cpu(),
            future_paths=future_paths.cpu(),
        )

    def adjust_path(self, query_sequence: torch.Tensor, base_path: torch.Tensor) -> Tuple[torch.Tensor, PatternMatch]:
        if not self.ready():
            return base_path, PatternMatch(patterns=[])
        device = next(self.encoder.parameters()).device
        self.encoder.eval()
        with torch.no_grad():
            query = query_sequence.unsqueeze(0).to(device).float()
            query_embedding = self.encoder(query).cpu().squeeze(0)
        patterns = self.database.search(query_embedding, k=self.k_retrieve) if self.database is not None else []
        if not patterns:
            return base_path, PatternMatch(patterns=[])
        weights = torch.tensor([p.similarity for p in patterns], dtype=torch.float32, device=base_path.device)
        weights = F.softmax(weights, dim=0)
        retrieved_paths = torch.stack([p.future_path[: base_path.shape[0]].to(base_path.device) for p in patterns], dim=0)
        blended_future = torch.sum(retrieved_paths * weights.view(-1, 1, 1), dim=0)
        adjusted = (1.0 - RAG_BLEND_WEIGHT) * base_path + RAG_BLEND_WEIGHT * blended_future
        match = PatternMatch(
            patterns=patterns,
            avg_similarity=float(weights.mean().item()),
            max_similarity=float(weights.max().item()),
            min_similarity=float(weights.min().item()),
        )
        return adjusted, match


## Integrated Model Definition (Phase 3 + 4 + 5)

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 0:
            pe[:, 1::2] = torch.cos(position * div_term)
        else:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)


class iTransformerEncoderLayer(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dim_feedforward: int = 512, dropout: float = 0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, src: torch.Tensor) -> torch.Tensor:
        batch, n_var, time_len, d_model = src.shape
        reshaped = src.permute(0, 2, 1, 3).reshape(batch * time_len, n_var, d_model)
        normed = self.norm1(reshaped)
        attn_out, _ = self.self_attn(normed, normed, normed, need_weights=False)
        reshaped = reshaped + self.dropout1(attn_out)
        src = reshaped.reshape(batch, time_len, n_var, d_model).permute(0, 2, 1, 3)
        src2 = self.norm2(src)
        src2 = self.linear2(self.dropout(F.gelu(self.linear1(src2))))
        return src + self.dropout2(src2)


class iTransformerEncoder(nn.Module):
    def __init__(self, input_size: int, d_model: int = 128, n_heads: int = 8, n_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.input_projection = nn.Linear(1, d_model)
        self.pos_encoder = SinusoidalPositionalEncoding(d_model, max_len=5000, dropout=dropout)
        self.layers = nn.ModuleList([iTransformerEncoderLayer(d_model, n_heads, dropout=dropout) for _ in range(n_layers)])

    def forward(self, src: torch.Tensor) -> torch.Tensor:
        batch_size, time_len, n_features = src.shape
        src = src.transpose(1, 2).unsqueeze(-1)
        src = self.input_projection(src)
        src = src.reshape(batch_size * n_features, time_len, -1)
        src = self.pos_encoder(src)
        src = src.reshape(batch_size, n_features, time_len, -1)
        for layer in self.layers:
            src = layer(src)
        return src.mean(dim=2).mean(dim=1)


class FrequencyFeatureExtractor(nn.Module):
    def __init__(self, n_fft: int = 16, hop_length: int = 4, out_channels: int = 64):
        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.n_freq_bins = n_fft // 2 + 1
        self.conv1 = nn.Conv1d(self.n_freq_bins, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, n_features = x.shape
        freq_features = []
        for feat_idx in range(min(n_features, 4)):
            signal = x[:, :, feat_idx]
            if seq_len < self.n_fft:
                signal = F.pad(signal, (0, self.n_fft - seq_len), mode='reflect')
            stft_out = torch.stft(
                signal,
                n_fft=self.n_fft,
                hop_length=self.hop_length,
                win_length=self.n_fft,
                window=torch.hann_window(self.n_fft, device=x.device),
                center=False,
                return_complex=True,
            )
            freq_features.append(torch.log1p(torch.abs(stft_out)))
        freq_features = torch.stack(freq_features, dim=0).mean(dim=0)
        out = self.conv1(freq_features)
        out = self.bn1(out)
        out = F.gelu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = F.gelu(out)
        return self.global_pool(out).squeeze(-1)


class MultiScaleFrequencyExtractor(nn.Module):
    def __init__(self, n_ffts: list[int] | None = None, out_channels: int = 64):
        super().__init__()
        self.n_ffts = n_ffts or [8, 16, 32]
        self.extractors = nn.ModuleList(
            [
                FrequencyFeatureExtractor(n_fft=n_fft, hop_length=max(1, n_fft // 4), out_channels=out_channels)
                for n_fft in self.n_ffts
            ]
        )
        self.fusion = nn.Sequential(
            nn.Linear(out_channels * len(self.n_ffts), out_channels * 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(out_channels * 2, out_channels),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        multi_scale = torch.cat([extractor(x) for extractor in self.extractors], dim=-1)
        return self.fusion(multi_scale)


class HybridFTEncoder(nn.Module):
    def __init__(
        self,
        input_size: int,
        hidden_size: int = 256,
        d_model: int = 128,
        n_heads: int = 8,
        n_layers: int = 2,
        num_gru_layers: int = 2,
        dropout: float = 0.1,
        use_frequency: bool = True,
        freq_out_channels: int = 64,
    ):
        super().__init__()
        self.use_frequency = use_frequency
        self.itransformer = iTransformerEncoder(
            input_size=input_size,
            d_model=d_model,
            n_heads=n_heads,
            n_layers=n_layers,
            dropout=dropout,
        )
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_gru_layers,
            batch_first=True,
            dropout=dropout if num_gru_layers > 1 else 0.0,
        )
        self.freq_encoder = MultiScaleFrequencyExtractor(
            n_ffts=MULTISCALE_N_FFTS,
            out_channels=freq_out_channels,
        ) if use_frequency else None
        fusion_input = d_model + hidden_size + (freq_out_channels if use_frequency else 0)
        self.fusion = nn.Sequential(
            nn.Linear(fusion_input, hidden_size * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 2, hidden_size),
        )
        self.output_norm = nn.LayerNorm(hidden_size)

    def forward(self, src: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        itrans_out = self.itransformer(src)
        gru_out, gru_hidden = self.gru(src)
        parts = [itrans_out, gru_hidden[-1]]
        if self.use_frequency and self.freq_encoder is not None:
            parts.append(self.freq_encoder(src))
        fused = self.fusion(torch.cat(parts, dim=-1))
        fused = self.output_norm(fused)
        return fused, gru_out


class Seq2SeqAttnGRU(nn.Module):
    def __init__(self, input_size, output_size, hidden_size, num_layers, dropout, horizon):
        super().__init__()
        self.horizon = horizon
        self.output_size = output_size
        self.hidden_size = hidden_size
        self.return_clip_low: Optional[torch.Tensor] = None
        self.return_clip_high: Optional[torch.Tensor] = None
        self.sigma_cap: Optional[torch.Tensor] = None
        self.rag_retriever: Optional[RAGPatternRetriever] = None
        self.last_rag_match: Optional[PatternMatch] = None

        self.encoder = HybridFTEncoder(
            input_size=input_size,
            hidden_size=hidden_size,
            d_model=D_MODEL,
            n_heads=N_HEADS,
            n_layers=N_LAYERS,
            num_gru_layers=num_layers,
            dropout=dropout,
            use_frequency=USE_FREQUENCY,
            freq_out_channels=FREQ_OUT_CHANNELS,
        )
        self.decoder_cell = nn.GRUCell(output_size + hidden_size, hidden_size)
        self.attn_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.mu_head = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, output_size),
        )
        self.log_sigma_head = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size // 2),
            nn.GELU(),
            nn.Linear(hidden_size // 2, output_size),
        )

        nn.init.xavier_uniform_(self.mu_head[-1].weight, gain=0.1)
        nn.init.zeros_(self.mu_head[-1].bias)
        nn.init.zeros_(self.log_sigma_head[-1].weight)
        nn.init.zeros_(self.log_sigma_head[-1].bias)

    def _bound_log_sigma(self, log_sigma: torch.Tensor) -> torch.Tensor:
        return torch.clamp(log_sigma, min=LOG_SIGMA_MIN, max=LOG_SIGMA_MAX)

    def set_prediction_constraints(self, clip_low: np.ndarray, clip_high: np.ndarray, sigma_cap: np.ndarray) -> None:
        self.return_clip_low = torch.as_tensor(clip_low, dtype=torch.float32)
        self.return_clip_high = torch.as_tensor(clip_high, dtype=torch.float32)
        self.sigma_cap = torch.as_tensor(sigma_cap, dtype=torch.float32)

    def _clip_returns(self, values: torch.Tensor, widen: float = 1.0) -> torch.Tensor:
        if self.return_clip_low is None or self.return_clip_high is None:
            return values
        low = self.return_clip_low.to(values.device).view(1, -1)
        high = self.return_clip_high.to(values.device).view(1, -1)
        if widen != 1.0:
            center = 0.5 * (low + high)
            half = 0.5 * (high - low) * widen
            low = center - half
            high = center + half
        return torch.maximum(torch.minimum(values, high), low)

    def _cap_sigma(self, sigma: torch.Tensor, historical_vol: Optional[float]) -> torch.Tensor:
        if self.sigma_cap is not None:
            sigma = torch.minimum(sigma, self.sigma_cap.to(sigma.device).view(1, -1))
        if historical_vol is not None:
            hist_cap = max(float(historical_vol) * 3.0, MIN_PREDICTED_VOL)
            sigma = torch.minimum(sigma, torch.full_like(sigma, hist_cap))
        return torch.maximum(sigma, torch.full_like(sigma, MIN_PREDICTED_VOL))

    def attach_retriever(self, retriever: RAGPatternRetriever) -> None:
        self.rag_retriever = retriever

    def _attend(self, h_dec, enc_out):
        query = self.attn_proj(h_dec).unsqueeze(2)
        scores = torch.bmm(enc_out, query).squeeze(2)
        weights = torch.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), enc_out).squeeze(1)
        return context

    def forward(self, x, y_teacher=None, teacher_forcing_ratio=0.0, return_sigma=False):
        h_dec, enc_out = self.encoder(x)
        dec_input = x[:, -1, : self.output_size]
        mu_seq, sigma_seq = [], []
        for t in range(self.horizon):
            context = self._attend(h_dec, enc_out)
            cell_input = torch.cat([dec_input, context], dim=1)
            h_dec = self.decoder_cell(cell_input, h_dec)
            out_features = torch.cat([h_dec, context], dim=1)
            mu = self._clip_returns(self.mu_head(out_features), widen=1.0)
            log_sigma = self._bound_log_sigma(self.log_sigma_head(out_features))
            mu_seq.append(mu.unsqueeze(1))
            sigma_seq.append(log_sigma.unsqueeze(1))
            if y_teacher is not None and teacher_forcing_ratio > 0.0:
                if teacher_forcing_ratio >= 1.0 or torch.rand(1).item() < teacher_forcing_ratio:
                    dec_input = y_teacher[:, t, :]
                else:
                    dec_input = mu.detach()
            else:
                dec_input = mu
        mu_out = torch.cat(mu_seq, dim=1)
        sigma_out = torch.cat(sigma_seq, dim=1)
        if return_sigma:
            return mu_out, sigma_out
        return mu_out

    def generate_realistic(self, x, temperature=1.0, historical_vol=None, manual_seed=None):
        self.eval()
        with torch.no_grad():
            if manual_seed is not None:
                torch.manual_seed(manual_seed)
            h_dec, enc_out = self.encoder(x)
            dec_input = x[:, -1, : self.output_size]
            generated = []
            for t in range(self.horizon):
                context = self._attend(h_dec, enc_out)
                cell_input = torch.cat([dec_input, context], dim=1)
                h_dec = self.decoder_cell(cell_input, h_dec)
                out_features = torch.cat([h_dec, context], dim=1)
                mu = self._clip_returns(self.mu_head(out_features), widen=1.10)
                log_sigma = self._bound_log_sigma(self.log_sigma_head(out_features))
                sigma = self._cap_sigma(torch.exp(log_sigma) * max(float(temperature), 0.0), historical_vol)
                if float(temperature) <= 0.0:
                    noise = torch.zeros_like(mu)
                else:
                    noise = torch.randn_like(mu) * sigma
                sample = self._clip_returns(mu + noise, widen=1.15)
                generated.append(sample.unsqueeze(1))
                dec_input = sample
            generated_paths = torch.cat(generated, dim=1)
            if self.rag_retriever is None or not self.rag_retriever.ready():
                self.last_rag_match = None
                return generated_paths
            adjusted_paths = []
            last_match = None
            for batch_idx in range(generated_paths.size(0)):
                adjusted_path, last_match = self.rag_retriever.adjust_path(
                    query_sequence=x[batch_idx].detach(),
                    base_path=generated_paths[batch_idx].detach(),
                )
                adjusted_paths.append(self._clip_returns(adjusted_path.to(generated_paths.device), widen=1.15))
            self.last_rag_match = last_match
            return torch.stack(adjusted_paths, dim=0)


### V10 Model Validation

In [ ]:
print('=' * 60)
print('V10 MODEL SMOKE TEST')
print('=' * 60)

test_device = RAG_DEVICE if LOW_VRAM_MODE else DEVICE
test_batch_size = 2 if LOW_VRAM_MODE else 4
test_seq_len = 96
test_input_size = len(BASE_FEATURE_COLS) + 1
test_horizon = 12

test_model = Seq2SeqAttnGRU(
    input_size=test_input_size,
    output_size=len(TARGET_COLS),
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    horizon=test_horizon,
).to(test_device)

test_input = torch.randn(test_batch_size, test_seq_len, test_input_size).to(test_device)
test_target = torch.randn(test_batch_size, test_horizon, len(TARGET_COLS)).to(test_device)

mu_out = test_model(test_input)
assert mu_out.shape == (test_batch_size, test_horizon, len(TARGET_COLS))
mu_out, sigma_out = test_model(test_input, return_sigma=True)
assert sigma_out.shape == (test_batch_size, test_horizon, len(TARGET_COLS))
gen_out = test_model.generate_realistic(test_input, temperature=1.0)
assert gen_out.shape == (test_batch_size, test_horizon, len(TARGET_COLS))

print({
    'forward_shape': tuple(mu_out.shape),
    'sigma_shape': tuple(sigma_out.shape),
    'generated_shape': tuple(gen_out.shape),
    'test_device': str(test_device),
})

test_model = test_model.to('cpu')
del test_model, test_input, test_target, mu_out, sigma_out, gen_out
cuda_cleanup()


## Trend Injection Functions (v7.5)

In [ ]:
def calculate_trend_slope(prices: np.ndarray) -> float:
    """
    Calculate linear trend slope from price series.
    Returns slope in log-return space per step.
    """
    prices = np.asarray(prices, dtype=np.float32)
    if len(prices) < 2:
        return 0.0

    # Use log prices for trend calculation
    log_prices = np.log(prices)
    x = np.arange(len(log_prices), dtype=np.float32)

    # Simple linear regression: slope = covariance(x, y) / variance(x)
    x_mean = np.mean(x)
    y_mean = np.mean(log_prices)

    numerator = np.sum((x - x_mean) * (log_prices - y_mean))
    denominator = np.sum((x - x_mean) ** 2)

    if denominator < 1e-10:
        return 0.0

    return float(numerator / denominator)


def calculate_path_trend(return_ohlc: np.ndarray) -> float:
    """
    Calculate trend slope from predicted OHLC returns.
    Uses Close prices for trend calculation.
    """
    # Convert returns to close prices (relative to starting point)
    close_returns = return_ohlc[:, 3]  # rClose column
    close_prices = np.exp(np.cumsum(close_returns))
    return calculate_trend_slope(close_prices)


def select_best_path_by_trend(
    all_paths: list[np.ndarray], 
    historical_slope: float,
    strong_trend_threshold: float = 0.002
) -> tuple[int, np.ndarray, dict]:
    """
    Select the path whose trend slope is closest to historical trend.

    Args:
        all_paths: List of generated paths, each [horizon, 4]
        historical_slope: Historical trend slope from input context
        strong_trend_threshold: Threshold for considering trend 'strong'

    Returns:
        best_idx: Index of selected path
        best_path: The selected path
        info: Dict with selection details for visualization
    """
    path_slopes = []
    path_directions = []

    for path in all_paths:
        slope = calculate_path_trend(path)
        path_slopes.append(slope)
        path_directions.append(np.sign(slope))

    path_slopes = np.array(path_slopes)
    path_directions = np.array(path_directions)

    # Check if historical trend is strong
    historical_direction = np.sign(historical_slope)
    is_strong_trend = abs(historical_slope) > strong_trend_threshold

    # Filter paths if strong trend
    if is_strong_trend:
        # Reject paths that go opposite direction
        valid_mask = path_directions == historical_direction
        if not valid_mask.any():
            # If all paths rejected, fall back to all paths
            valid_mask = np.ones(len(all_paths), dtype=bool)
    else:
        valid_mask = np.ones(len(all_paths), dtype=bool)

    # Calculate distance to historical slope for valid paths
    slope_distances = np.abs(path_slopes - historical_slope)
    slope_distances[~valid_mask] = np.inf  # Exclude invalid paths

    # Select path with minimum distance
    best_idx = int(np.argmin(slope_distances))
    best_path = all_paths[best_idx]

    # Build info dict for visualization
    info = {
        'historical_slope': historical_slope,
        'historical_direction': historical_direction,
        'is_strong_trend': is_strong_trend,
        'strong_threshold': strong_trend_threshold,
        'path_slopes': path_slopes.tolist(),
        'path_directions': path_directions.tolist(),
        'valid_mask': valid_mask.tolist(),
        'selected_idx': best_idx,
        'selected_slope': path_slopes[best_idx],
        'rejected_count': int((~valid_mask).sum()),
        'slope_distances': slope_distances[valid_mask].tolist() if valid_mask.any() else [],
    }

    return best_idx, best_path, info


def generate_ensemble_with_trend_selection(
    model, 
    X: np.ndarray,
    context_prices: np.ndarray,
    temperature: float = 1.0,
    ensemble_size: int = 20,
    trend_lookback: int = 20
) -> tuple[np.ndarray, dict]:
    """
    Generate ensemble of paths and select best matching historical trend.

    Args:
        model: Trained Seq2SeqAttnGRU model
        X: Input features [1, window, features]
        context_prices: Historical price context for volatility and trend calc
        temperature: Sampling temperature
        ensemble_size: Number of paths to generate
        trend_lookback: Number of bars to use for trend calculation

    Returns:
        best_path: Selected path [horizon, 4]
        ensemble_info: Dict with all paths and selection info
    """
    model.eval()

    with torch.no_grad():
        X_tensor = torch.from_numpy(X).float().to(DEVICE)

        # Calculate historical volatility from context
        log_returns = np.log(context_prices[1:] / context_prices[:-1])
        historical_vol = float(np.std(log_returns)) if len(log_returns) > 1 else 0.001

        # Calculate historical trend from last N bars of context
        trend_context = context_prices[-trend_lookback:]
        historical_slope = calculate_trend_slope(trend_context)

        print(f"Historical realized vol: {historical_vol:.6f}, Temperature: {temperature}")
        print(f"Historical trend slope (last {trend_lookback} bars): {historical_slope:.6f}")

        # Generate ensemble of paths with different seeds
        all_paths = []
        for i in range(ensemble_size):
            # Use different seed for each path
            seed = SEED + i * 1000
            generated = model.generate_realistic(
                X_tensor, 
                temperature=temperature, 
                historical_vol=historical_vol,
                manual_seed=seed
            )
            all_paths.append(generated.detach().cpu().numpy()[0])  # [horizon, 4]

        # Select best path by trend matching
        best_idx, best_path, selection_info = select_best_path_by_trend(
            all_paths, 
            historical_slope,
            STRONG_TREND_THRESHOLD
        )

        print(f"Generated {ensemble_size} paths, selected path {best_idx}")
        print(f"  Selected path slope: {selection_info['selected_slope']:.6f}")
        print(f"  Strong trend: {selection_info['is_strong_trend']}, Rejected: {selection_info['rejected_count']}")

        # Build ensemble info for visualization
        ensemble_info = {
            'all_paths': all_paths,
            'best_idx': best_idx,
            'best_path': best_path,
            'selection_info': selection_info,
            'historical_vol': historical_vol,
            'temperature': temperature,
        }

        return best_path, ensemble_info

## Loss Functions

In [ ]:
def clamp_log_sigma(log_sigma):
    return torch.clamp(log_sigma, min=LOG_SIGMA_MIN, max=LOG_SIGMA_MAX)


def nll_loss(mu, log_sigma, target):
    """Per-step Gaussian negative log-likelihood."""
    bounded_log_sigma = clamp_log_sigma(log_sigma)
    sigma = torch.exp(bounded_log_sigma)
    return 0.5 * ((target - mu) / sigma) ** 2 + bounded_log_sigma + 0.5 * np.log(2 * np.pi)


def candle_range_loss(mu, target):
    pred_range = mu[:, :, 1] - mu[:, :, 2]  # High - Low
    actual_range = target[:, :, 1] - target[:, :, 2]
    return ((pred_range - actual_range) ** 2).mean()


def volatility_match_loss(mu, log_sigma, target):
    """Calibrate sigma to realized autoregressive forecast error."""
    pred_vol = torch.exp(clamp_log_sigma(log_sigma))
    realized_abs_error = (target - mu).detach().abs()
    return ((pred_vol - realized_abs_error) ** 2).mean()


def directional_penalty(mu, target):
    pred_close = mu[:, :, 3]
    actual_close = target[:, :, 3]
    mask = actual_close.abs() >= DIRECTION_EPS
    if not mask.any():
        return pred_close.new_tensor(0.0)
    sign_match = torch.sign(pred_close[mask]) * torch.sign(actual_close[mask])
    penalty = torch.clamp(-sign_match, min=0.0)
    return penalty.mean()


def compute_target_constraints(target_windows: np.ndarray) -> dict[str, np.ndarray]:
    flat = target_windows.reshape(-1, target_windows.shape[-1]).astype(np.float32)
    target_std = flat.std(axis=0).astype(np.float32)
    if APPLY_CLIPPING:
        clip_low = np.quantile(flat, CLIP_QUANTILES[0], axis=0).astype(np.float32)
        clip_high = np.quantile(flat, CLIP_QUANTILES[1], axis=0).astype(np.float32)
    else:
        clip_low = np.min(flat, axis=0).astype(np.float32)
        clip_high = np.max(flat, axis=0).astype(np.float32)
    sigma_cap = np.clip(target_std * 3.0, MIN_PREDICTED_VOL, 0.01).astype(np.float32)
    return {
        'clip_low': clip_low,
        'clip_high': clip_high,
        'target_std': target_std,
        'sigma_cap': sigma_cap,
    }


def apply_target_clipping(target_windows: np.ndarray, constraints: dict[str, np.ndarray]) -> np.ndarray:
    low = constraints['clip_low'].reshape(1, 1, -1)
    high = constraints['clip_high'].reshape(1, 1, -1)
    return np.clip(target_windows, low, high).astype(np.float32)


## Training Functions

In [ ]:
def tf_ratio_for_epoch(epoch):
    ratio = TF_START * (TF_DECAY_RATE ** (epoch - 1))
    return max(float(TF_END), float(ratio))


def run_epoch(model, loader, step_weights_t, optimizer=None, tf_ratio=0.0, device=None):
    is_train = optimizer is not None
    model.train(is_train)
    device = DEVICE if device is None else torch.device(device)

    total_loss, nll_total, range_total, vol_total, dir_total = 0, 0, 0, 0, 0
    n_items = 0

    for xb, yb_s, yb_r in loader:
        xb = xb.to(device)
        yb_s = yb_s.to(device)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            mu, log_sigma = model(
                xb,
                y_teacher=yb_s if is_train else None,
                teacher_forcing_ratio=tf_ratio if is_train else 0.0,
                return_sigma=True,
            )

            nll = (nll_loss(mu, log_sigma, yb_s) * step_weights_t).mean()
            rng = candle_range_loss(mu, yb_s)
            vol = volatility_match_loss(mu, log_sigma, yb_s)
            dir_pen = directional_penalty(mu, yb_s)

            loss = nll + RANGE_LOSS_WEIGHT * rng + VOLATILITY_WEIGHT * vol + DIR_PENALTY_WEIGHT * dir_pen

            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        bs = xb.size(0)
        total_loss += loss.item() * bs
        nll_total += nll.item() * bs
        range_total += rng.item() * bs
        vol_total += vol.item() * bs
        dir_total += dir_pen.item() * bs
        n_items += bs

    return {
        'total': total_loss / max(n_items, 1),
        'nll': nll_total / max(n_items, 1),
        'range': range_total / max(n_items, 1),
        'vol': vol_total / max(n_items, 1),
        'dir': dir_total / max(n_items, 1),
    }


In [ ]:
def train_model(model, train_loader, val_loader, max_epochs, patience, device=None):
    device = DEVICE if device is None else torch.device(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6)

    step_idx = np.arange(HORIZON, dtype=np.float32)
    step_w = 1.0 + (step_idx / max(HORIZON - 1, 1)) ** STEP_LOSS_POWER
    step_weights_t = torch.as_tensor(step_w, dtype=torch.float32, device=device).view(1, HORIZON, 1)

    best_val = float('inf')
    best_state = copy.deepcopy(model.state_dict())
    wait = 0
    rows = []

    for epoch in range(1, max_epochs + 1):
        tf = tf_ratio_for_epoch(epoch)
        tr = run_epoch(model, train_loader, step_weights_t, optimizer=optimizer, tf_ratio=tf, device=device)
        va = run_epoch(model, val_loader, step_weights_t, optimizer=None, tf_ratio=0.0, device=device)

        scheduler.step(va['total'])
        lr = optimizer.param_groups[0]['lr']

        rows.append({
            'epoch': epoch, 'tf_ratio': tf, 'lr': lr,
            'train_total': tr['total'], 'val_total': va['total'],
            'train_nll': tr['nll'], 'val_nll': va['nll'],
            'train_range': tr['range'], 'val_range': va['range'],
        })

        print(f"Epoch {epoch:02d} | tf={tf:.3f} | "
              f"train={tr['total']:.6f} (nll={tr['nll']:.6f}) | "
              f"val={va['total']:.6f} (nll={va['nll']:.6f}) | lr={lr:.6g} | device={device}")

        if va['total'] < best_val:
            best_val = va['total']
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f'Early stopping at epoch {epoch}.')
                break

    model.load_state_dict(best_state)
    return pd.DataFrame(rows)


## Evaluation Functions

In [ ]:
def evaluate_metrics(actual_ohlc, pred_ohlc, prev_close):
    actual_ohlc = np.asarray(actual_ohlc, dtype=np.float32)
    pred_ohlc = np.asarray(pred_ohlc, dtype=np.float32)
    ac, pc = actual_ohlc[:, 3], pred_ohlc[:, 3]

    return {
        'close_mae': float(np.mean(np.abs(ac - pc))),
        'close_rmse': float(np.sqrt(np.mean((ac - pc) ** 2))),
        'ohlc_mae': float(np.mean(np.abs(actual_ohlc - pred_ohlc))),
        'directional_accuracy_eps': float(np.mean(np.sign(ac - prev_close) == np.sign(pc - prev_close))),
    }

def evaluate_baselines(actual_ohlc, prev_ohlc, prev_close):
    persistence = evaluate_metrics(actual_ohlc, prev_ohlc, prev_close)
    flat = np.repeat(prev_close.reshape(-1, 1), 4, axis=1).astype(np.float32)
    flat_rw = evaluate_metrics(actual_ohlc, flat, prev_close)
    return {'persistence': persistence, 'flat_close_rw': flat_rw}

## Main Training Function

In [ ]:
def run_fold(fold_name, price_fold, window, max_epochs, patience, run_sanity=False, quick_mode=False):
    feat_fold = build_feature_frame(price_fold)
    target_fold = build_target_frame(feat_fold)

    input_raw = feat_fold[BASE_FEATURE_COLS].to_numpy(np.float32)
    target_raw = target_fold[TARGET_COLS].to_numpy(np.float32)
    row_imputed = feat_fold['row_imputed'].to_numpy(np.int8).astype(bool)
    row_open_skip = feat_fold['row_open_skip'].to_numpy(np.int8).astype(bool)
    prev_close = feat_fold['prev_close'].to_numpy(np.float32)
    price_vals = price_fold.loc[feat_fold.index, OHLC_COLS].to_numpy(np.float32)

    tr_end, va_end = split_points(len(input_raw))

    # Standardize inputs only
    in_mean, in_std = input_raw[:tr_end].mean(axis=0), input_raw[:tr_end].std(axis=0)
    in_std = np.where(in_std < 1e-8, 1.0, in_std)
    input_scaled = (input_raw - in_mean) / in_std

    # No target scaling (raw returns)
    tg_mean, tg_std = np.zeros(4, dtype=np.float32), np.ones(4, dtype=np.float32)
    target_scaled = target_raw.copy()

    X_all, y_all_s, y_all_r, starts, prev_close_starts, dropped_imputed, dropped_skip = make_multistep_windows(
        input_scaled, target_scaled, target_raw, row_imputed, row_open_skip, prev_close, window, HORIZON
    )

    if len(X_all) == 0:
        raise RuntimeError(f'{fold_name}: no windows available.')

    # Splits
    end_idx = starts + HORIZON - 1
    tr_m, va_m, te_m = end_idx < tr_end, (end_idx >= tr_end) & (end_idx < va_end), end_idx >= va_end

    X_train, y_train_s, y_train_r = X_all[tr_m], y_all_s[tr_m], y_all_r[tr_m]
    X_val, y_val_s, y_val_r = X_all[va_m], y_all_s[va_m], y_all_r[va_m]
    X_test, y_test_s, y_test_r = X_all[te_m], y_all_s[te_m], y_all_r[te_m]
    test_starts = starts[te_m]
    test_prev_close = prev_close_starts[te_m]

    target_constraints = compute_target_constraints(y_train_r)
    if APPLY_CLIPPING:
        y_train_s = apply_target_clipping(y_train_s, target_constraints)
        y_val_s = apply_target_clipping(y_val_s, target_constraints)

    print(f'Samples: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}')

    # Data loaders
    train_loader = DataLoader(MultiStepDataset(X_train, y_train_s, y_train_r), 
                             batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(MultiStepDataset(X_val, y_val_s, y_val_r), 
                           batch_size=BATCH_SIZE, shuffle=False)

    # Model
    model = Seq2SeqAttnGRU(
        input_size=X_train.shape[-1],
        output_size=len(TARGET_COLS),
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        horizon=HORIZON,
    ).to(DEVICE)
    model.set_prediction_constraints(
        target_constraints['clip_low'],
        target_constraints['clip_high'],
        target_constraints['sigma_cap'],
    )

    hist = train_model(model, train_loader, val_loader, max_epochs, patience)

    rag_retriever = RAGPatternRetriever(
        input_size=X_train.shape[-1],
        embedding_dim=RAG_EMBEDDING_DIM,
        k_retrieve=RAG_K_RETRIEVE,
        hidden_size=RAG_ENCODER_HIDDEN,
        num_layers=RAG_ENCODER_LAYERS,
    ).to(RAG_DEVICE)
    rag_limit = min(RAG_MAX_PATTERNS, len(X_train))
    rag_retriever.build_database(
        sequences=torch.from_numpy(X_train[:rag_limit]).float(),
        future_paths=torch.from_numpy(y_train_r[:rag_limit]).float(),
        batch_size=RAG_BUILD_BATCH_SIZE,
    )
    model.attach_retriever(rag_retriever)


    # ENSEMBLE TREND INJECTION PREDICTION (v7.5)
    # Pick last test sample for visualization
    last_idx = len(X_test) - 1
    X_last = X_test[last_idx:last_idx+1]
    context_start = int(test_starts[last_idx]) - window
    context_prices = price_vals[context_start:int(test_starts[last_idx]), 3]  # Close prices

    # Generate ensemble with trend selection
    best_rets, ensemble_info = generate_ensemble_with_trend_selection(
        model, X_last, context_prices, 
        temperature=SAMPLING_TEMPERATURE,
        ensemble_size=ENSEMBLE_SIZE,
        trend_lookback=TREND_LOOKBACK_BARS
    )

    # Convert to prices
    last_close = float(test_prev_close[last_idx])
    pred_price_best = returns_to_prices_seq(best_rets, last_close)

    # Convert all paths to prices for visualization
    all_price_paths = []
    for path_rets in ensemble_info['all_paths']:
        path_prices = returns_to_prices_seq(path_rets, last_close)
        all_price_paths.append(path_prices)
    ensemble_info['all_price_paths'] = all_price_paths

    # Actual future
    actual_future = price_vals[int(test_starts[last_idx]):int(test_starts[last_idx])+HORIZON]

    # One-step metrics (deterministic for comparison)
    mu_test = model(torch.from_numpy(X_test).float().to(DEVICE)).detach().cpu().numpy()
    pred_step1_ret = mu_test[:, 0, :]
    actual_step1_ret = y_test_r[:, 0, :]

    # Simple price reconstruction for metrics
    pred_ohlc_1 = np.zeros((len(test_starts), 4))
    for i in range(len(test_starts)):
        pc = test_prev_close[i]
        pred_ohlc_1[i] = [
            pc * np.exp(pred_step1_ret[i, 0]),
            pc * np.exp(pred_step1_ret[i, 1]),
            pc * np.exp(pred_step1_ret[i, 2]),
            pc * np.exp(pred_step1_ret[i, 3]),
        ]
        pred_ohlc_1[i] = enforce_candle_validity(pred_ohlc_1[i].reshape(1, -1))[0]

    actual_ohlc_1 = price_vals[test_starts]
    prev_ohlc = price_vals[test_starts - 1]

    model_metrics = evaluate_metrics(actual_ohlc_1, pred_ohlc_1, test_prev_close)
    baseline_metrics = evaluate_baselines(actual_ohlc_1, prev_ohlc, test_prev_close)

    print(f"\nEnsemble prediction stats:")
    print(f"  Pred range (best): [{pred_price_best[:, 3].min():.2f}, {pred_price_best[:, 3].max():.2f}]")
    print(f"  Actual range: [{actual_future[:, 3].min():.2f}, {actual_future[:, 3].max():.2f}]")
    print(f"  Pred volatility (best): {np.std(best_rets[:, 3]):.6f}")
    print(f"  Actual volatility: {np.std(actual_step1_ret[:, 3]):.6f}")

    # Build DataFrames for plotting
    future_idx = price_fold.index[test_starts[last_idx]:test_starts[last_idx]+HORIZON]
    pred_future_df = pd.DataFrame(pred_price_best, index=future_idx, columns=OHLC_COLS)
    actual_future_df = pd.DataFrame(actual_future, index=future_idx, columns=OHLC_COLS)
    context_df = price_fold.iloc[test_starts[last_idx]-window:test_starts[last_idx]][OHLC_COLS]

    model = model.to('cpu')
    cuda_cleanup()

    return {
        'fold': fold_name,
        'window': window,
        'history_df': hist,
        'model_metrics': model_metrics,
        'baseline_metrics': baseline_metrics,
        'context_df': context_df,
        'actual_future_df': actual_future_df,
        'pred_future_df': pred_future_df,
        'ensemble_info': ensemble_info,
        'samples': {'train': len(X_train), 'val': len(X_val), 'test': len(X_test)},
        'rag_database_size': rag_retriever.database.size() if rag_retriever.database is not None else 0,
    }

## Run Experiments

In [ ]:
# Run lookback sweep if enabled
fold_results = []
primary_slice = slices[0]
selected_window = DEFAULT_LOOKBACK

if ENABLE_LOOKBACK_SWEEP:
    print('\n=== Lookback sweep ===')
    _, a0, b0 = primary_slice
    fold_price0 = price_df.iloc[a0:b0].copy()

    best_score = -float('inf')
    for w in LOOKBACK_CANDIDATES:
        print(f'\nSweep candidate lookback={w} --')
        try:
            r = run_fold(f'sweep_w{w}', fold_price0, w, SWEEP_MAX_EPOCHS, SWEEP_PATIENCE, quick_mode=True)
            score = -r['model_metrics']['close_mae']  # Simple selection
            if score > best_score:
                best_score = score
                selected_window = w
        except Exception as e:
            print(f"Failed for window {w}: {e}")

print(f'\nSelected lookback: {selected_window}')

In [ ]:
# Run full walk-forward with ensemble trend injection
print('\n=== Full walk-forward with ensemble trend injection ===')
for i, (name, a, b) in enumerate(slices, start=1):
    print(f'\n=== Running {name} [{a}:{b}] lookback={selected_window} ===')
    fold_price = price_df.iloc[a:b].copy()
    try:
        res = run_fold(name, fold_price, selected_window, FINAL_MAX_EPOCHS, FINAL_PATIENCE)
        fold_results.append(res)

        print(f"\nResults for {name}:")
        print(f"  Model MAE: {res['model_metrics']['close_mae']:.4f}")
        print(f"  Persistence MAE: {res['baseline_metrics']['persistence']['close_mae']:.4f}")
    except Exception as e:
        print(f"Error in fold {name}: {e}")

## Visualization with Momentum Confidence

In [ ]:
if fold_results:
    latest = fold_results[-1]
    ensemble_info = latest['ensemble_info']
    selection_info = ensemble_info['selection_info']

    fig, axes = plt.subplots(2, 1, figsize=(18, 12), facecolor='black', 
                            gridspec_kw={'height_ratios': [2, 1]})
    ax_main = axes[0]
    ax_trend = axes[1]
    ax_main.set_facecolor('black')
    ax_trend.set_facecolor('black')

    def draw_candles(ax, ohlc, start_x, up_edge, up_face, down_edge, down_face, wick_color, width=0.6, alpha=1.0):
        vals = ohlc[OHLC_COLS].to_numpy()
        for i, (o, h, l, c) in enumerate(vals):
            x = start_x + i
            bull = c >= o
            ax.vlines(x, l, h, color=wick_color, linewidth=1.0, alpha=alpha, zorder=2)
            lower = min(o, c)
            height = max(abs(c - o), 1e-6)
            rect = Rectangle((x - width/2, lower), width, height,
                           facecolor=up_face if bull else down_face,
                           edgecolor=up_edge if bull else down_edge,
                           linewidth=1.0, alpha=alpha, zorder=3)
            ax.add_patch(rect)

    def draw_path_line(ax, price_path, start_x, color, alpha=0.3, linewidth=1, label=None):
        """Draw a simple line path showing close prices"""
        closes = price_path[:, 3]  # Close column
        x_vals = np.arange(len(closes)) + start_x
        ax.plot(x_vals, closes, color=color, alpha=alpha, linewidth=linewidth, label=label)

    context_df = latest['context_df']
    actual_future_df = latest['actual_future_df']
    pred_future_df = latest['pred_future_df']

    # Draw history (green/red)
    draw_candles(ax_main, context_df, 0, '#00FF00', '#00FF00', '#FF0000', '#FF0000', '#FFFFFF', width=0.6, alpha=0.9)

    # Draw all ensemble paths as faint lines
    all_price_paths = ensemble_info['all_price_paths']
    best_idx = ensemble_info['best_idx']

    for i, path_prices in enumerate(all_price_paths):
        if i == best_idx:
            continue  # Skip best path, will draw separately
        # Color based on whether path was valid
        if selection_info['valid_mask'][i]:
            color = '#444444'  # Valid but not selected
            alpha = 0.2
        else:
            color = '#330000'  # Rejected (wrong direction)
            alpha = 0.15
        draw_path_line(ax_main, path_prices, len(context_df), color, alpha=alpha, linewidth=1)

    # Draw best path (bright)
    draw_path_line(ax_main, all_price_paths[best_idx], len(context_df), '#00FFFF', alpha=0.8, 
                   linewidth=2, label=f'Best Path (#{best_idx})')

    # Draw actual future (dimmed candles)
    draw_candles(ax_main, actual_future_df, len(context_df), '#00AA00', '#00AA00', '#AA0000', '#AA0000', '#888888', 
                 width=0.6, alpha=0.6)

    # Draw selected prediction (bright candles)
    draw_candles(ax_main, pred_future_df, len(context_df), '#00FFFF', '#00FFFF', '#0088AA', '#0088AA', '#00FFFF',
                 width=0.5, alpha=0.9)

    ax_main.axvline(len(context_df) - 0.5, color='white', linestyle='--', linewidth=1.0, alpha=0.8)

    # Labels for main chart
    n = len(context_df) + len(actual_future_df)
    step = max(1, n // 12)
    ticks = list(range(0, n, step))
    all_idx = context_df.index.append(actual_future_df.index)
    labels = [all_idx[i].strftime('%m-%d %H:%M') for i in ticks if i < len(all_idx)]

    ax_main.set_xticks(ticks)
    ax_main.set_xticklabels(labels, rotation=30, ha='right', color='white', fontsize=9)
    ax_main.tick_params(axis='y', colors='white')
    for sp in ax_main.spines.values():
        sp.set_color('#666666')
    ax_main.grid(color='#333333', linewidth=0.5, alpha=0.5)

    title_text = (f"MSFT 1m ({latest['fold']}) - Ensemble Trend Injection | "
                  f"Temp={SAMPLING_TEMPERATURE}, Ensemble={ENSEMBLE_SIZE} | "
                  f"Selected Path: #{best_idx}")
    ax_main.set_title(title_text, color='white', fontsize=14, pad=15)
    ax_main.set_ylabel('Price', color='white', fontsize=12)

    # Legend for main chart
    legend_elements = [
        Patch(facecolor='#00FF00', edgecolor='#00FF00', label='History (bull)'),
        Patch(facecolor='#FF0000', edgecolor='#FF0000', label='History (bear)'),
        Patch(facecolor='#00AA00', edgecolor='#00AA00', label='Actual Future'),
        Patch(facecolor='#00FFFF', edgecolor='#00FFFF', label='Selected Prediction'),
    ]
    ax_main.legend(handles=legend_elements, facecolor='black', edgecolor='white', labelcolor='white', 
                  loc='upper left', fontsize=10)

    # === Trend Analysis Subplot ===
    path_slopes = selection_info['path_slopes']
    historical_slope = selection_info['historical_slope']
    valid_mask = selection_info['valid_mask']

    # Plot histogram of path slopes
    valid_slopes = [s for s, v in zip(path_slopes, valid_mask) if v]
    rejected_slopes = [s for s, v in zip(path_slopes, valid_mask) if not v]

    if valid_slopes:
        ax_trend.hist(valid_slopes, bins=15, color='#00AA00', alpha=0.6, label='Valid Paths')
    if rejected_slopes:
        ax_trend.hist(rejected_slopes, bins=10, color='#AA0000', alpha=0.4, label='Rejected Paths')

    # Mark historical slope
    ax_trend.axvline(historical_slope, color='#FFFF00', linestyle='-', linewidth=2, 
                    label=f'Historical Slope: {historical_slope:.6f}')

    # Mark selected path slope
    selected_slope = selection_info['selected_slope']
    ax_trend.axvline(selected_slope, color='#00FFFF', linestyle='--', linewidth=2,
                    label=f'Selected Slope: {selected_slope:.6f}')

    # Mark strong trend threshold if applicable
    if selection_info['is_strong_trend']:
        thresh = selection_info['strong_threshold']
        if historical_slope > 0:
            ax_trend.axvspan(-thresh*2, -thresh, alpha=0.2, color='red', label='Rejection Zone')
        else:
            ax_trend.axvspan(thresh, thresh*2, alpha=0.2, color='red', label='Rejection Zone')

    ax_trend.set_xlabel('Trend Slope (log-return per step)', color='white', fontsize=11)
    ax_trend.set_ylabel('Path Count', color='white', fontsize=11)
    ax_trend.set_title('Momentum Confidence: Trend Slope Distribution', color='white', fontsize=12)
    ax_trend.tick_params(axis='x', colors='white')
    ax_trend.tick_params(axis='y', colors='white')
    for sp in ax_trend.spines.values():
        sp.set_color('#666666')
    ax_trend.legend(facecolor='black', edgecolor='white', labelcolor='white', loc='upper right')
    ax_trend.grid(color='#333333', linewidth=0.5, alpha=0.5)

    plt.tight_layout()
    plt.show()

    # Print selection details
    print(f"\n=== Momentum Confidence Report ===")
    print(f"Historical trend (last {TREND_LOOKBACK_BARS} bars): {historical_slope:.6f}")
    print(f"Strong trend detected: {selection_info['is_strong_trend']}")
    print(f"Paths rejected (wrong direction): {selection_info['rejected_count']}/{ENSEMBLE_SIZE}")
    print(f"Selected path index: {best_idx}")
    print(f"Selected path slope: {selected_slope:.6f}")
    print(f"Slope difference: {abs(selected_slope - historical_slope):.6f}")
    print(f"\nAll path slopes: {[f'{s:.4f}' for s in path_slopes]}")

## Test Cell: Validation Metrics

In [ ]:
# Test Cell: Print validation metrics and visual confirmation
print("=" * 60)
print("VALIDATION TEST CELL - Forecast and Rolling Backtest Metrics")
print("=" * 60)

if fold_results:
    for result in fold_results:
        fold_name = result['fold']
        print(f"\n--- Results for {fold_name} ---")

        # Directional accuracy vs persistence baseline
        model_dir_acc = result['model_metrics']['directional_accuracy_eps']
        persist_dir_acc = result['baseline_metrics']['persistence']['directional_accuracy_eps']
        print(f"Directional Accuracy:")
        print(f"  Model:           {model_dir_acc:.4f} ({model_dir_acc*100:.2f}%)")
        print(f"  Persistence:     {persist_dir_acc:.4f} ({persist_dir_acc*100:.2f}%)")
        print(f"  Improvement:     {(model_dir_acc - persist_dir_acc):+.4f} ({(model_dir_acc - persist_dir_acc)*100:+.2f}%)")

        # Average predicted close vs actual close (bias)
        actual_future = result['actual_future_df']
        pred_future = result['pred_future_df']
        actual_closes = actual_future['Close'].values
        pred_closes = pred_future['Close'].values

        avg_actual = np.mean(actual_closes)
        avg_pred = np.mean(pred_closes)
        bias = avg_pred - avg_actual

        print(f"\nAverage Close Bias:")
        print(f"  Actual avg:      ${avg_actual:.4f}")
        print(f"  Predicted avg:   ${avg_pred:.4f}")
        print(f"  Bias:            ${bias:.4f} ({bias/avg_actual*100:+.4f}%)")

        # Visual confirmation of candle realism
        print(f"\nCandle Realism Check (Selected Path):")
        ohlc_data = pred_future[OHLC_COLS].values

        # Check candle validity
        valid_candles = 0
        total_wicks = []
        total_bodies = []

        for o, h, l, c in ohlc_data:
            # High >= max(Open, Close) and Low <= min(Open, Close)
            is_valid = (h >= max(o, c)) and (l <= min(o, c))
            if is_valid:
                valid_candles += 1

            # Calculate wicks and body
            upper_wick = h - max(o, c)
            lower_wick = min(o, c) - l
            body = abs(c - o)
            total_wicks.append(upper_wick + lower_wick)
            total_bodies.append(body)

        validity_rate = valid_candles / len(ohlc_data)
        avg_wick = np.mean(total_wicks)
        avg_body = np.mean(total_bodies)

        print(f"  Valid candles:   {valid_candles}/{len(ohlc_data)} ({validity_rate*100:.1f}%)")
        print(f"  Avg wick size:   ${avg_wick:.4f}")
        print(f"  Avg body size:   ${avg_body:.4f}")
        print(f"  Wick/Body ratio: {avg_wick/avg_body:.2f}")

        # Ensemble diversity
        ensemble_info = result['ensemble_info']
        all_paths = ensemble_info['all_price_paths']
        final_closes = [path[-1, 3] for path in all_paths]  # Close price at final step
        close_std = np.std(final_closes)
        close_range = max(final_closes) - min(final_closes)

        print(f"\nEnsemble Diversity:")
        print(f"  Final close std: ${close_std:.4f}")
        print(f"  Final close range: ${close_range:.4f}")
        print(f"  Selection info:  Path #{ensemble_info['best_idx']} selected by trend matching")

print("\n" + "=" * 60)
print("VALIDATION COMPLETE")
print("=" * 60)

## V8 Extension: Rolling Walk-Forward Backtest (Strictly Causal)
This section adds a full bar-by-bar rolling engine on top of the v7 training pipeline above.

Important:
- Model architecture is unchanged (`Seq2SeqAttnGRU` + `generate_realistic`).
- No checkpoint is required; the model is trained in this notebook first.
- Rolling predictions are strictly causal at anchor time `t` using only `[t-window, ..., t-1]`.

In [ ]:
# V8 rolling configuration (frame generator mode)
ROLLINGSTARTTIME = '09:30'
ROLLINGENDTIME = '16:00'
ROLLING_STEP = 1  # 1 = every minute

DEFAULT_ROLLING_TEMPERATURE = 0.45
BASE_ROLLING_TEMPERATURE = DEFAULT_ROLLING_TEMPERATURE
USE_TEMPERATURE_SCHEDULE = True
TEMPERATURESCHEDULE = [
    ('09:30', '10:15', 0.35),
    ('10:15', '14:00', 0.45),
    ('14:00', '16:00', 0.55),
]

ROLLING_BACKTEST_DATE = None  # e.g. '2025-02-13'

FRAME_OUTPUT_DIR = Path('output/jupyter-notebook/frames/v10')
FRAME_FILENAME_PATTERN = 'frame_{:04d}.png'
FRAME_DPI = 180
FRAME_FIGSIZE = (18, 8)
FRAME_HISTORY_BARS = 220

print({
    'ROLLINGSTARTTIME': ROLLINGSTARTTIME,
    'ROLLINGENDTIME': ROLLINGENDTIME,
    'ROLLING_STEP': ROLLING_STEP,
    'DEFAULT_ROLLING_TEMPERATURE': DEFAULT_ROLLING_TEMPERATURE,
    'USE_TEMPERATURE_SCHEDULE': USE_TEMPERATURE_SCHEDULE,
    'ROLLING_BACKTEST_DATE': ROLLING_BACKTEST_DATE,
    'FRAME_OUTPUT_DIR': str(FRAME_OUTPUT_DIR),
    'FRAME_DPI': FRAME_DPI,
})


In [ ]:
# Train a v7 model for rolling backtest (no checkpoint needed)
def _intraday_positions_for_date(df: pd.DataFrame, date_str: str, start_time: str, end_time: str) -> np.ndarray:
    idx = df.index
    day_mask = (idx.strftime('%Y-%m-%d') == date_str)
    st = pd.Timestamp(start_time).time()
    et = pd.Timestamp(end_time).time()
    time_mask = np.array([(t >= st) and (t < et) for t in idx.time], dtype=bool)
    return np.where(day_mask & time_mask)[0]


def _select_backtest_date(df: pd.DataFrame, requested: Optional[str], window: int, horizon: int) -> str:
    if requested is not None:
        pos = _intraday_positions_for_date(df, requested, ROLLINGSTARTTIME, ROLLINGENDTIME)
        if len(pos) == 0:
            raise ValueError(f'No intraday bars for requested date: {requested}')
        if pos[0] < window:
            raise ValueError(f'Requested date {requested} does not have enough prior bars for window={window}')
        if pos[-1] + horizon >= len(df):
            raise ValueError(f'Requested date {requested} lacks future bars for horizon={horizon}')
        return requested

    # Auto-pick latest valid date excluding final date to ensure future bars for full 390 anchors.
    dates = sorted(pd.Index(df.index.strftime('%Y-%m-%d')).unique())
    if len(dates) < 2:
        raise RuntimeError('Need at least 2 trading dates for rolling backtest with full horizon labels.')

    for d in reversed(dates[:-1]):
        pos = _intraday_positions_for_date(df, d, ROLLINGSTARTTIME, ROLLINGENDTIME)
        if len(pos) < 390:
            continue
        if pos[0] < window:
            continue
        if pos[-1] + horizon >= len(df):
            continue
        return d

    raise RuntimeError('Could not auto-select a valid backtest date with full intraday coverage.')


def train_v7_model_for_rolling(
    price_df_full: pd.DataFrame,
    window: int,
    horizon: int,
    backtest_date: str,
) -> Tuple[nn.Module, pd.DataFrame, np.ndarray, np.ndarray, pd.DataFrame]:
    feat_all = build_feature_frame(price_df_full)
    target_all = build_target_frame(feat_all)

    all_dates = feat_all.index.strftime('%Y-%m-%d')
    train_day_mask = all_dates < backtest_date
    if train_day_mask.sum() < (window + horizon + 500):
        raise RuntimeError(
            f'Not enough pre-backtest bars before {backtest_date}: {train_day_mask.sum()} rows.'
        )

    input_raw = feat_all[BASE_FEATURE_COLS].to_numpy(np.float32)
    target_raw = target_all[TARGET_COLS].to_numpy(np.float32)
    row_imputed = feat_all['row_imputed'].to_numpy(np.int8).astype(bool)
    row_open_skip = feat_all['row_open_skip'].to_numpy(np.int8).astype(bool)
    prev_close = feat_all['prev_close'].to_numpy(np.float32)

    # Fit scaler using ONLY pre-backtest rows (causal-safe scaling).
    pre_idx = np.where(train_day_mask)[0]
    fit_end = int(pre_idx[-1]) + 1

    in_mean = input_raw[:fit_end].mean(axis=0)
    in_std = input_raw[:fit_end].std(axis=0)
    in_std = np.where(in_std < 1e-8, 1.0, in_std)
    input_scaled = ((input_raw - in_mean) / in_std).astype(np.float32)

    # v7 target setup (no target scaling)
    target_scaled = target_raw.copy()

    X_all, y_all_s, y_all_r, starts, prev_close_starts, drop_imp, drop_skip = make_multistep_windows(
        input_scaled=input_scaled,
        target_scaled=target_scaled,
        target_raw=target_raw,
        row_imputed=row_imputed,
        row_open_skip=row_open_skip,
        starts_prev_close=prev_close,
        window=window,
        horizon=horizon,
    )

    if len(X_all) == 0:
        raise RuntimeError('No windows available after filtering.')

    # Keep windows whose prediction horizon end is strictly before backtest date.
    end_idx = starts + horizon - 1
    end_dates = feat_all.index[end_idx].strftime('%Y-%m-%d')
    usable = end_dates < backtest_date

    X_all = X_all[usable]
    y_all_s = y_all_s[usable]
    y_all_r = y_all_r[usable]

    if len(X_all) < 1000:
        raise RuntimeError(f'Not enough usable windows before backtest date ({len(X_all)}).')

    split = int(len(X_all) * 0.85)
    X_train, y_train_s, y_train_r = X_all[:split], y_all_s[:split], y_all_r[:split]
    X_val, y_val_s, y_val_r = X_all[split:], y_all_s[split:], y_all_r[split:]

    target_constraints = compute_target_constraints(y_train_r)
    if APPLY_CLIPPING:
        y_train_s = apply_target_clipping(y_train_s, target_constraints)
        y_val_s = apply_target_clipping(y_val_s, target_constraints)

    train_loader = DataLoader(MultiStepDataset(X_train, y_train_s, y_train_r), batch_size=ROLLING_TRAIN_BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(MultiStepDataset(X_val, y_val_s, y_val_r), batch_size=ROLLING_TRAIN_BATCH_SIZE, shuffle=False)

    model = Seq2SeqAttnGRU(
        input_size=X_train.shape[-1],
        output_size=len(TARGET_COLS),
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        horizon=horizon,
    ).to(ROLLING_TRAIN_DEVICE)
    model.set_prediction_constraints(
        target_constraints['clip_low'],
        target_constraints['clip_high'],
        target_constraints['sigma_cap'],
    )

    print({
        'train_windows': len(X_train),
        'val_windows': len(X_val),
        'dropped_target_imputed': int(drop_imp),
        'dropped_target_open_skip': int(drop_skip),
        'backtest_date': backtest_date,
        'rolling_train_device': str(ROLLING_TRAIN_DEVICE),
        'rolling_train_batch_size': ROLLING_TRAIN_BATCH_SIZE,
    })


    history_df = train_model(model, train_loader, val_loader, max_epochs=FINAL_MAX_EPOCHS, patience=FINAL_PATIENCE, device=ROLLING_TRAIN_DEVICE)

    model = model.to('cpu')
    cuda_cleanup()

    rag_retriever = RAGPatternRetriever(
        input_size=X_all.shape[-1],
        embedding_dim=RAG_EMBEDDING_DIM,
        k_retrieve=RAG_K_RETRIEVE,
        hidden_size=RAG_ENCODER_HIDDEN,
        num_layers=RAG_ENCODER_LAYERS,
    ).to(RAG_DEVICE)
    rag_limit = min(ROLLING_RAG_MAX_PATTERNS, len(X_all))
    rag_retriever.build_database(
        sequences=torch.from_numpy(X_all[:rag_limit]).float(),
        future_paths=torch.from_numpy(y_all_r[:rag_limit]).float(),
        batch_size=RAG_BUILD_BATCH_SIZE,
    )
    model.attach_retriever(rag_retriever)
    print({'rag_database_size': rag_retriever.database.size() if rag_retriever.database is not None else 0})
    return model, feat_all, in_mean.astype(np.float32), in_std.astype(np.float32), history_df


ROLLING_BACKTEST_DATE = _select_backtest_date(price_df, ROLLING_BACKTEST_DATE, DEFAULT_LOOKBACK, HORIZON)
print(f'Selected rolling backtest date: {ROLLING_BACKTEST_DATE}')

rolling_model, rolling_feat_df, rolling_in_mean, rolling_in_std, rolling_train_history = train_v7_model_for_rolling(
    price_df_full=price_df,
    window=DEFAULT_LOOKBACK,
    horizon=HORIZON,
    backtest_date=ROLLING_BACKTEST_DATE,
)

print('Rolling model trained and ready.')

In [ ]:
@dataclass
class RollingPredictionLog:
    """Enhanced rolling prediction log with regime information."""
    anchortime: pd.Timestamp
    contextendprice: float
    predictedpath: pd.DataFrame
    actualpath: pd.DataFrame
    predictionhorizon: int
    base_temperature: float
    regime: MarketRegime
    regime_temp_mult: float
    adjusted_temperature: float
    context_start_idx: int
    context_end_idx: int
    turbulence: float = 0.0
    atr: float = 0.0

    step_mae: Optional[np.ndarray] = None
    directional_hit: Optional[bool] = None

    def __post_init__(self):
        assert len(self.predictedpath) == self.predictionhorizon
        assert len(self.actualpath) == self.predictionhorizon
        assert self.predictedpath.index[0] == self.anchortime
        assert self.actualpath.index[0] == self.anchortime

    def compute_metrics(self):
        p = self.predictedpath['Close'].to_numpy(np.float32)
        a = self.actualpath['Close'].to_numpy(np.float32)
        self.step_mae = np.abs(p - a)
        self.directional_hit = bool(np.sign(p[0] - self.contextendprice) == np.sign(a[0] - self.contextendprice))
        return self


class RollingBacktester:
    """Phase 2: Regime-aware rolling backtester."""

    def __init__(
        self,
        model: nn.Module,
        pricedf: pd.DataFrame,
        featuredf: pd.DataFrame,
        input_mean: np.ndarray,
        input_std: np.ndarray,
        windowsize: int,
        horizon: int,
        regime_config: RegimeConfig = None,
    ):
        self.model = model.to(DEVICE)
        self.model.eval()
        self.pricedf = pricedf.copy()
        self.featuredf = featuredf.copy()
        self.input_mean = input_mean.astype(np.float32)
        self.input_std = np.where(input_std.astype(np.float32) < 1e-8, 1.0, input_std.astype(np.float32))
        self.windowsize = int(windowsize)
        self.horizon = int(horizon)

        # Phase 2: Initialize regime detector
        self.regime_detector = MarketRegimeDetector(regime_config or REGIME_CONFIG)

        self.input_raw = self.featuredf[BASE_FEATURE_COLS].to_numpy(np.float32)
        self.input_scaled = ((self.input_raw - self.input_mean) / self.input_std).astype(np.float32)
        self.row_imputed = self.featuredf['row_imputed'].to_numpy(np.int8).astype(bool)
        self.returns = self.featuredf['returns'].to_numpy(np.float32)
        self.atr_values = self.featuredf['atr_14'].to_numpy(np.float32)

        self.ts_to_pos = {ts: i for i, ts in enumerate(self.featuredf.index)}
        self.day_anchor_positions: Optional[np.ndarray] = None
        self.selected_anchor_positions: Optional[np.ndarray] = None

        # Track regime statistics
        self.regime_counts = {MarketRegime.NORMAL: 0, MarketRegime.ELEVATED: 0, MarketRegime.CRISIS: 0}

    def _hist_vol(self, context_start: int, context_end: int) -> float:
        closes = self.pricedf['Close'].iloc[context_start:context_end].to_numpy(np.float32)
        if len(closes) < 2:
            return 0.001
        lr = np.log(closes[1:] / np.maximum(closes[:-1], 1e-8))
        return max(float(np.std(lr)), MIN_PREDICTED_VOL)

    @torch.no_grad()
    def _predict_path(self, context_start: int, context_end: int, temperature: float) -> np.ndarray:
        assert context_end - context_start == self.windowsize
        x_raw = self.input_scaled[context_start:context_end]
        imp_frac = float(self.row_imputed[context_start:context_end].mean())
        imp_col = np.full((self.windowsize, 1), imp_frac, dtype=np.float32)
        x_aug = np.concatenate([x_raw, imp_col], axis=1)
        x_tensor = torch.from_numpy(x_aug).unsqueeze(0).float().to(DEVICE)

        hist_vol = self._hist_vol(context_start, context_end)
        pred_ret = self.model.generate_realistic(x_tensor, temperature=temperature, historical_vol=hist_vol)[0]
        return pred_ret.detach().cpu().numpy()

    def runrollingbacktest(
        self, starttime: str, endtime: str, date: str, step: int = 1
    ) -> Tuple[List[RollingPredictionLog], int]:
        idx = self.featuredf.index
        st = pd.Timestamp(starttime).time()
        en = pd.Timestamp(endtime).time()
        mask = (idx.strftime('%Y-%m-%d') == date) & np.array(
            [(t >= st) and (t < en) for t in idx.time], dtype=bool
        )
        anchor_positions = np.where(mask)[0]

        if len(anchor_positions) == 0:
            raise RuntimeError(f'No anchors for date={date} {starttime}-{endtime}')

        selected = anchor_positions[::step]
        self.day_anchor_positions = anchor_positions
        self.selected_anchor_positions = selected

        logs: List[RollingPredictionLog] = []

        # Phase 2: Reset regime detector for each backtest
        self.regime_detector = MarketRegimeDetector(REGIME_CONFIG)
        self.regime_counts = {MarketRegime.NORMAL: 0, MarketRegime.ELEVATED: 0, MarketRegime.CRISIS: 0}

        for k, anchor_pos in enumerate(selected, start=1):
            context_end = int(anchor_pos)
            context_start = context_end - self.windowsize
            if context_start < 0:
                continue

            day_idx = np.searchsorted(anchor_positions, anchor_pos)
            valid_steps = min(self.horizon, len(anchor_positions) - day_idx)
            if valid_steps <= 0:
                continue

            prediction_time = idx[context_end]
            context = self.featuredf.iloc[context_start:context_end]

            # Causality check
            assert context.index[-1] < prediction_time

            # Phase 2: Detect regime at prediction time
            current_return = np.array([self.returns[context_end]])
            current_atr = float(self.atr_values[context_end])

            regime = self.regime_detector.detect_regime(
                returns=current_return,
                atr=current_atr,
                timestamp=prediction_time
            )
            self.regime_counts[regime] += 1

            # Phase 2: Get regime-adjusted temperature
            temp_mult = self.regime_detector.get_temperature_multiplier()
            adjusted_temp = BASE_ROLLING_TEMPERATURE * temp_mult

            pred_rets_full = self._predict_path(context_start, context_end, adjusted_temp)
            pred_rets = pred_rets_full[:valid_steps]

            context_close = float(self.featuredf['prev_close'].iloc[context_end])
            pred_prices = returns_to_prices_seq(pred_rets, context_close)

            future_positions = anchor_positions[day_idx: day_idx + valid_steps]
            pred_index = idx[future_positions]
            pred_df = pd.DataFrame(pred_prices, index=pred_index, columns=OHLC_COLS)
            actual_df = self.pricedf[OHLC_COLS].iloc[future_positions].copy()

            log = RollingPredictionLog(
                anchortime=prediction_time,
                contextendprice=context_close,
                predictedpath=pred_df,
                actualpath=actual_df,
                predictionhorizon=valid_steps,
                base_temperature=BASE_ROLLING_TEMPERATURE,
                regime=regime,
                regime_temp_mult=temp_mult,
                adjusted_temperature=adjusted_temp,
                context_start_idx=context_start,
                context_end_idx=context_end,
                turbulence=self.regime_detector._turbulence_history[-1] if self.regime_detector._turbulence_history else 0.0,
                atr=current_atr,
            ).compute_metrics()
            logs.append(log)

        expected_count = len(selected)
        return logs, expected_count

    def get_regime_summary(self) -> dict:
        """Get summary of regime distribution during backtest."""
        total = sum(self.regime_counts.values())
        if total == 0:
            return {}
        return {
            'total_predictions': total,
            'normal_count': self.regime_counts[MarketRegime.NORMAL],
            'normal_pct': self.regime_counts[MarketRegime.NORMAL] / total * 100,
            'elevated_count': self.regime_counts[MarketRegime.ELEVATED],
            'elevated_pct': self.regime_counts[MarketRegime.ELEVATED] / total * 100,
            'crisis_count': self.regime_counts[MarketRegime.CRISIS],
            'crisis_pct': self.regime_counts[MarketRegime.CRISIS] / total * 100,
            'regime_transitions': len(self.regime_detector.get_regime_history()),
        }

def runrollingbacktest(model, pricedf, windowsize, starttime, endtime):
    rb = RollingBacktester(
        model=model,
        pricedf=pricedf,
        featuredf=rolling_feat_df,
        input_mean=rolling_in_mean,
        input_std=rolling_in_std,
        windowsize=windowsize,
        horizon=HORIZON,
        regime_config=REGIME_CONFIG,
    )
    return rb.runrollingbacktest(starttime=starttime, endtime=endtime, date=ROLLING_BACKTEST_DATE, step=ROLLING_STEP), rb


In [ ]:
# Run rolling backtest + required validation asserts
(rolling_logs, expected_prediction_count), rolling_backtester = runrollingbacktest(
    model=rolling_model,
    pricedf=price_df,
    windowsize=DEFAULT_LOOKBACK,
    starttime=ROLLINGSTARTTIME,
    endtime=ROLLINGENDTIME,
)

if len(rolling_logs) == 0:
    raise RuntimeError('No rolling logs produced.')

# Required assert 1: first prediction timestamp equals anchor timestamp
assert rolling_logs[0].predictedpath.index[0] == rolling_logs[0].anchortime, (
    'first prediction first candle timestamp does not equal prediction anchor time'
)

# Required assert 2: no prediction uses future context
for log in rolling_logs:
    anchor_pos = rolling_backtester.ts_to_pos[log.anchortime]
    assert log.context_end_idx == anchor_pos, 'context_end_idx must equal anchor position'
    assert (log.context_end_idx - 1) < anchor_pos, 'context must strictly end at t-1'

# Required assert 3: prediction count equals expected minute anchors for this run
assert len(rolling_logs) == expected_prediction_count, (
    f'prediction count mismatch: {len(rolling_logs)} != {expected_prediction_count}'
)

print({
    'rolling_date': ROLLING_BACKTEST_DATE,
    'predictions_generated': len(rolling_logs),
    'expected_prediction_count': expected_prediction_count,
    'first_anchor': str(rolling_logs[0].anchortime),
    'first_prediction_first_ts': str(rolling_logs[0].predictedpath.index[0]),
    'last_horizon_steps': rolling_logs[-1].predictionhorizon,
})
if hasattr(rolling_backtester, 'get_regime_summary'):
    print({'regime_summary': rolling_backtester.get_regime_summary()})


In [ ]:
def _draw_candles(
    ax,
    ohlc_df: pd.DataFrame,
    start_x: int,
    up_edge: str,
    up_face: str,
    down_edge: str,
    down_face: str,
    wick_color: str,
    width: float = 0.58,
    lw: float = 1.0,
    alpha: float = 1.0,
):
    vals = ohlc_df[OHLC_COLS].to_numpy(np.float32)
    for i, (o, h, l, c) in enumerate(vals):
        x = start_x + i
        bull = c >= o
        ax.vlines(x, l, h, color=wick_color, linewidth=lw, alpha=alpha, zorder=2)
        lower = min(o, c)
        height = max(abs(c - o), 1e-6)
        rect = Rectangle(
            (x - width / 2, lower),
            width,
            height,
            facecolor=up_face if bull else down_face,
            edgecolor=up_edge if bull else down_edge,
            linewidth=lw,
            alpha=alpha,
            zorder=3,
        )
        ax.add_patch(rect)


def _format_ts(ts: pd.Timestamp) -> str:
    return ts.strftime('%I:%M %p').lstrip('0')


def render_single_frame(
    log: RollingPredictionLog,
    frame_idx: int,
    total_frames: int,
    pricedf: pd.DataFrame,
    backtester: RollingBacktester,
    history_bars: int = FRAME_HISTORY_BARS,
) -> plt.Figure:
    """Render frame with Phase 2 regime indicator."""
    anchor_pos = backtester.ts_to_pos[log.anchortime]
    h_start = max(0, anchor_pos - history_bars)
    history_df = pricedf.iloc[h_start:anchor_pos][OHLC_COLS].copy()

    actual_df = log.actualpath.copy()
    pred_df = log.predictedpath.copy()

    fig, ax = plt.subplots(figsize=FRAME_FIGSIZE, facecolor='black')
    FigureCanvasAgg(fig)
    ax.set_facecolor('black')

    # Historical context
    _draw_candles(ax, history_df, 0,
                  up_edge='#00FF00', up_face='#00FF00',
                  down_edge='#FF0000', down_face='#FF0000',
                  wick_color='#D0D0D0', width=0.60, lw=1.0, alpha=0.95)

    # Actual future
    future_start_x = len(history_df)
    _draw_candles(ax, actual_df, future_start_x,
                  up_edge='#1D6F42', up_face='#1D6F42',
                  down_edge='#8E2F25', down_face='#8E2F25',
                  wick_color='#8E8E8E', width=0.58, lw=0.9, alpha=0.40)

    # Predicted future
    _draw_candles(ax, pred_df, future_start_x,
                  up_edge='#FFFFFF', up_face='#FFFFFF',
                  down_edge='#FFFFFF', down_face='#000000',
                  wick_color='#F3F3F3', width=0.50, lw=1.2, alpha=1.0)

    # NOW divider
    now_x = len(history_df) - 0.5
    ax.axvline(now_x, color='white', linestyle='--', linewidth=1.0, alpha=0.85, zorder=4)
    ax.text(now_x + 0.8, ax.get_ylim()[1] if len(ax.get_ylim()) == 2 else 0.0, 'NOW',
            color='white', fontsize=9)

    # Phase 2: Regime indicator box
    regime_color = log.regime.color
    regime_name = log.regime.display_name
    ax.text(
        0.99, 0.97,
        f'REGIME: {regime_name}\nTemp: {log.base_temperature:.2f} Ã— {log.regime_temp_mult:.1f} = {log.adjusted_temperature:.2f}',
        transform=ax.transAxes,
        va='top', ha='right',
        color=regime_color,
        fontsize=11,
        fontweight='bold',
        bbox=dict(
            facecolor='black',
            edgecolor=regime_color,
            alpha=0.9,
            boxstyle='round,pad=0.4'
        ),
    )

    # Axes
    full_idx = history_df.index.append(actual_df.index)
    n = len(full_idx)
    step = max(1, n // 10)
    ticks = list(range(0, n, step))
    if ticks[-1] != n - 1:
        ticks.append(n - 1)
    labels = [full_idx[i].strftime('%H:%M') for i in ticks]

    ax.set_xticks(ticks)
    ax.set_xticklabels(labels, rotation=25, ha='right', color='white', fontsize=9)
    ax.tick_params(axis='y', colors='white')
    for sp in ax.spines.values():
        sp.set_color('#666666')

    ax.grid(color='#242424', linewidth=0.6, alpha=0.35)

    header = (
        f"{SYMBOL} 1m | Timestamp: {_format_ts(log.anchortime)} | "
        f"Frame {frame_idx + 1}/{total_frames} | {regime_name} Regime"
    )
    ax.set_title(header, color='white', pad=12)
    ax.set_ylabel('Price', color='white')

    # Frame counter
    ax.text(
        0.01, 0.99,
        f'Frame {frame_idx + 1}/{total_frames}',
        transform=ax.transAxes,
        va='top', ha='left', color='white', fontsize=10,
        bbox=dict(facecolor='black', edgecolor='#666666', alpha=0.8, boxstyle='round,pad=0.25'),
    )

    legend_elements = [
        Patch(facecolor='#00FF00', edgecolor='#00FF00', label='History (bull)'),
        Patch(facecolor='#FF0000', edgecolor='#FF0000', label='History (bear)'),
        Patch(facecolor='#1D6F42', edgecolor='#1D6F42', label='Actual Future'),
        Patch(facecolor='#FFFFFF', edgecolor='#FFFFFF', label='Predicted (bull)'),
        Patch(facecolor='#000000', edgecolor='#FFFFFF', label='Predicted (bear)'),
    ]
    leg = ax.legend(handles=legend_elements, facecolor='black', edgecolor='#666666', loc='upper left')
    for t in leg.get_texts():
        t.set_color('white')

    plt.tight_layout()
    return fig


def generate_rolling_frames(
    logs: List[RollingPredictionLog],
    pricedf: pd.DataFrame,
    backtester: RollingBacktester,
    output_dir: Path
) -> List[Path]:
    """Generate frames with regime visualization."""
    output_dir.mkdir(parents=True, exist_ok=True)
    total = len(logs)
    saved_paths: List[Path] = []

    for i, log in enumerate(logs):
        fig = render_single_frame(
            log=log,
            frame_idx=i,
            total_frames=total,
            pricedf=pricedf,
            backtester=backtester,
            history_bars=FRAME_HISTORY_BARS,
        )

        out_path = output_dir / FRAME_FILENAME_PATTERN.format(i)
        fig.savefig(out_path, dpi=FRAME_DPI, facecolor='black', bbox_inches='tight')
        saved_paths.append(out_path)
        plt.close('all')

    return saved_paths

output_dir = Path("./frames")
print(f"Number of logs: {len(rolling_logs)}")
saved = generate_rolling_frames(
    logs=rolling_logs,
    pricedf=price_df,
    backtester=rolling_backtester,
    output_dir=output_dir,
)
print(f"Frames saved: {len(saved)}")

## Regime Validation

In [ ]:
print("=" * 60)
print("PHASE 2: REGIME DETECTION TEST CELL")
print("=" * 60)

# Test 1: TurbulenceIndexCalculator
print("\n--- Test 1: TurbulenceIndexCalculator ---")
calc = TurbulenceIndexCalculator(lookback=20)

# Normal returns - low volatility
normal_returns = np.random.randn(50) * 0.01
for r in normal_returns:
    calc.update(np.array([r]))

# Crisis returns - high volatility
crisis_returns = np.random.randn(20) * 0.08
crisis_turbulence = []
for r in crisis_returns:
    t = calc.update(np.array([r]))
    crisis_turbulence.append(t)

print(f'Normal period avg turbulence: {np.mean([calc.update(np.array([r])) for r in normal_returns[-10:]]):.4f}')
print(f'Crisis period avg turbulence: {np.mean(crisis_turbulence):.4f}')
print(f'Turbulence increases during crisis: {np.mean(crisis_turbulence) > np.mean([calc.update(np.array([r])) for r in normal_returns[-10:]])}')

# Test 2: MarketRegimeDetector
print("\n--- Test 2: MarketRegimeDetector ---")
detector = MarketRegimeDetector(RegimeConfig(
    normal_threshold=0.75,
    elevated_threshold=0.90,
    normal_temp_mult=1.0,
    elevated_temp_mult=1.3,
    crisis_temp_mult=1.8,
))

# Simulate normal market
print("Simulating normal market conditions...")
for i in range(70):
    r = np.random.randn() * 0.01
    atr = 0.01
    regime = detector.detect_regime(np.array([r]), atr)

print(f'Regime after normal period: {regime}')
print(f'Temperature multiplier: {detector.get_temperature_multiplier()}')

# Simulate elevated volatility
print("\nSimulating elevated volatility...")
for i in range(20):
    r = np.random.randn() * 0.03
    atr = 0.03
    regime = detector.detect_regime(np.array([r]), atr)

print(f'Regime after elevated period: {regime}')
print(f'Temperature multiplier: {detector.get_temperature_multiplier()}')

# Simulate crisis
print("\nSimulating crisis conditions...")
for i in range(10):
    r = np.random.randn() * 0.08 - 0.05  # High vol with negative drift
    atr = 0.08
    regime = detector.detect_regime(np.array([r]), atr)

print(f'Regime after crisis period: {regime}')
print(f'Temperature multiplier: {detector.get_temperature_multiplier()}')
print(f'Should halt trading: {detector.should_halt_trading()}')

# Test 3: Regime transitions
print("\n--- Test 3: Regime Transition Logging ---")
history = detector.get_regime_history()
print(f'Number of regime transitions: {len(history)}')
for ts, reg in history[:5]:
    print(f'  {ts}: {reg}')

# Test 4: Temperature multipliers
print("\n--- Test 4: Temperature Multiplier Verification ---")
test_detector = MarketRegimeDetector()

test_detector._current_regime = MarketRegime.NORMAL
assert test_detector.get_temperature_multiplier() == 1.0, "NORMAL should have multiplier 1.0"
print("NORMAL regime: multiplier = 1.0 âœ“")

test_detector._current_regime = MarketRegime.ELEVATED
assert test_detector.get_temperature_multiplier() == 1.3, "ELEVATED should have multiplier 1.3"
print("ELEVATED regime: multiplier = 1.3 âœ“")

test_detector._current_regime = MarketRegime.CRISIS
assert test_detector.get_temperature_multiplier() == 1.8, "CRISIS should have multiplier 1.8"
print("CRISIS regime: multiplier = 1.8 âœ“")

print("\n" + "=" * 60)
print("REGIME DETECTION TESTS PASSED")
print("=" * 60)

## Phase 6: RL Decision Layer
This section uses the rolling forecast outputs as the state input to a PPO trading policy.
The price model and strictly causal rolling backtest are preserved from the V8.5 base notebook.


In [ ]:
@dataclass
class Trade:
    timestamp: int
    action: str
    shares: float
    price: float
    cost: float


class MarketRegime(Enum):
    NORMAL = 'normal'
    ELEVATED = 'elevated'
    CRISIS = 'crisis'


@dataclass
class RegimeConfig:
    normal_threshold: float = 0.75
    elevated_threshold: float = 0.90
    normal_position_mult: float = 1.0
    elevated_position_mult: float = 0.7
    crisis_position_mult: float = 0.3
    lookback: int = 60


class TurbulenceIndexCalculator:
    def __init__(self, lookback: int = 60):
        self.lookback = lookback
        self.history: List[np.ndarray] = []

    def update(self, returns: np.ndarray) -> float:
        self.history.append(np.asarray(returns, dtype=np.float32))
        if len(self.history) < self.lookback:
            return 0.0
        history = np.asarray(self.history[-self.lookback :], dtype=np.float32)
        mean = history.mean(axis=0)
        cov = np.cov(history.T)
        if np.ndim(cov) < 2:
            cov = np.array([[cov]], dtype=np.float32)
        cov = cov + np.eye(cov.shape[0], dtype=np.float32) * 1e-6
        diff = np.asarray(returns, dtype=np.float32) - mean
        try:
            inv_cov = np.linalg.inv(cov)
            return float(np.sqrt(diff @ inv_cov @ diff))
        except np.linalg.LinAlgError:
            return float(np.linalg.norm(diff))


class MarketRegimeDetector:
    def __init__(self, config: RegimeConfig | None = None):
        self.config = config or RegimeConfig()
        self.turbulence_calc = TurbulenceIndexCalculator(self.config.lookback)
        self.atr_history: List[float] = []
        self.turbulence_history: List[float] = []
        self.current_regime = MarketRegime.NORMAL

    def detect_regime(self, returns: np.ndarray, atr: float) -> MarketRegime:
        turbulence = self.turbulence_calc.update(returns)
        self.atr_history.append(float(atr))
        self.atr_history = self.atr_history[-self.config.lookback :]
        self.turbulence_history.append(float(turbulence))
        self.turbulence_history = self.turbulence_history[-self.config.lookback :]
        if len(self.atr_history) < self.config.lookback:
            self.current_regime = MarketRegime.NORMAL
            return self.current_regime
        atr_percentile = np.mean(np.asarray(self.atr_history) <= atr)
        turb_values = np.asarray([t for t in self.turbulence_history if t > 0.0], dtype=np.float32)
        turb_percentile = np.mean(turb_values <= turbulence) if len(turb_values) else 0.5
        if atr_percentile > self.config.elevated_threshold and turb_percentile > self.config.elevated_threshold:
            self.current_regime = MarketRegime.CRISIS
        elif atr_percentile > self.config.normal_threshold or turb_percentile > self.config.normal_threshold:
            self.current_regime = MarketRegime.ELEVATED
        else:
            self.current_regime = MarketRegime.NORMAL
        return self.current_regime

    def get_position_multiplier(self) -> float:
        return {
            MarketRegime.NORMAL: self.config.normal_position_mult,
            MarketRegime.ELEVATED: self.config.elevated_position_mult,
            MarketRegime.CRISIS: self.config.crisis_position_mult,
        }[self.current_regime]

    def get_regime_indicator(self) -> float:
        return {
            MarketRegime.NORMAL: 0.0,
            MarketRegime.ELEVATED: 0.5,
            MarketRegime.CRISIS: 1.0,
        }[self.current_regime]


def build_rl_market_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    close = out['Close']
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = (-delta.clip(upper=0.0))
    avg_gain = gain.ewm(alpha=1 / 14, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / 14, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    out['rsi_14'] = 100.0 - (100.0 / (1.0 + rs))

    ema_fast = close.ewm(span=12, adjust=False).mean()
    ema_slow = close.ewm(span=26, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    macd_signal = macd_line.ewm(span=9, adjust=False).mean()
    out['macd_histogram'] = macd_line - macd_signal

    sma20 = close.rolling(20).mean()
    std20 = close.rolling(20).std()
    bb_upper = sma20 + 2.0 * std20
    bb_lower = sma20 - 2.0 * std20
    bb_width = (bb_upper - bb_lower).replace(0.0, np.nan)
    out['bb_position'] = (close - bb_lower) / bb_width

    high_low = out['High'] - out['Low']
    high_close = (out['High'] - out['Close'].shift()).abs()
    low_close = (out['Low'] - out['Close'].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    out['atr_14'] = tr.ewm(alpha=1 / 14, adjust=False).mean()
    out['atr_14_pct'] = out['atr_14'] / close.replace(0.0, np.nan)

    out['price_momentum_5'] = close / close.shift(5) - 1.0
    out['price_momentum_20'] = close / close.shift(20) - 1.0
    typical_price = (out['High'] + out['Low'] + out['Close']) / 3.0
    rolling_volume = out['Volume'].rolling(20).sum().replace(0.0, np.nan)
    out['vwap_20'] = (typical_price * out['Volume']).rolling(20).sum() / rolling_volume
    out['vwap_20_dev'] = (out['Close'] - out['vwap_20']) / out['vwap_20'].replace(0.0, np.nan)

    direction = np.sign(out['Close'].diff().fillna(0.0))
    out['direction'] = direction
    out['returns'] = close.pct_change().fillna(0.0)
    out['obv'] = (direction * out['Volume']).cumsum()
    out['obv_slope'] = out['obv'].diff(5)
    return out.replace([np.inf, -np.inf], np.nan).ffill().fillna(0.0)


def pad_prediction_paths(logs: List[RollingPredictionLog], horizon: int) -> np.ndarray:
    padded = []
    for log in logs:
        path = log.predictedpath[OHLC_COLS].to_numpy(np.float32)
        if len(path) < horizon:
            pad_rows = np.repeat(path[-1:], horizon - len(path), axis=0)
            path = np.concatenate([path, pad_rows], axis=0)
        padded.append(path[:horizon])
    return np.stack(padded).astype(np.float32)


class StockTradingEnv:
    def __init__(
        self,
        df: pd.DataFrame,
        predictions: np.ndarray,
        initial_balance: float = 100000.0,
        transaction_cost: float = 0.001,
        max_position: float = 1.0,
        lookback: int = 60,
        use_turbulence: bool = True,
        dsr_eta: float = 0.1,
    ):
        self.df = df.reset_index(drop=True).copy()
        self.predictions = predictions.astype(np.float32)
        self.initial_balance = float(initial_balance)
        self.transaction_cost = float(transaction_cost)
        self.max_position = float(max_position)
        self.lookback = int(min(lookback, max(1, len(df) - 2)))
        self.use_turbulence = bool(use_turbulence)
        self.dsr_eta = float(dsr_eta)
        self.regime_detector = MarketRegimeDetector() if use_turbulence else None
        self.state_dim = self.predictions.shape[1] * 4 + 10 + 4 + (1 if use_turbulence else 0)
        self.action_dim = 1
        self.reset()

    def reset(self) -> np.ndarray:
        self.balance = self.initial_balance
        self.shares = 0.0
        self.portfolio_value = self.initial_balance
        self.current_step = self.lookback
        self.trades: List[Trade] = []
        self.portfolio_history = [self.initial_balance]
        self.returns_history: List[float] = []
        self.A = 0.0
        self.B = 0.0
        if self.regime_detector is not None:
            self.regime_detector = MarketRegimeDetector(self.regime_detector.config)
        return self._get_observation()

    def _get_market_state(self) -> np.ndarray:
        row = self.df.iloc[self.current_step]
        vol_ma = self.df['Volume'].iloc[max(0, self.current_step - 19) : self.current_step + 1].mean()
        return np.asarray(
            [
                row['rsi_14'] / 100.0,
                row['macd_histogram'],
                row['bb_position'],
                row['atr_14_pct'],
                row['price_momentum_5'],
                row['price_momentum_20'],
                row['vwap_20_dev'],
                row['obv_slope'] / 1e6,
                row['direction'],
                row['Volume'] / vol_ma if vol_ma > 0 else 1.0,
            ],
            dtype=np.float32,
        )

    def _get_observation(self) -> np.ndarray:
        pred = self.predictions[self.current_step].reshape(-1).astype(np.float32)
        price = float(self.df['Close'].iloc[self.current_step])
        portfolio_value = self.balance + self.shares * price
        position_pct = (self.shares * price) / portfolio_value if portfolio_value else 0.0
        portfolio = np.asarray(
            [
                self.balance / self.initial_balance,
                self.shares / 1000.0,
                portfolio_value / self.initial_balance,
                position_pct,
            ],
            dtype=np.float32,
        )
        market = self._get_market_state()
        if self.regime_detector is not None:
            regime = np.asarray([self.regime_detector.get_regime_indicator()], dtype=np.float32)
            return np.concatenate([pred, market, portfolio, regime]).astype(np.float32)
        return np.concatenate([pred, market, portfolio]).astype(np.float32)

    def _calculate_dsr_reward(self, ret: float) -> float:
        if len(self.returns_history) < 2:
            return 0.0
        A_prev = self.A
        B_prev = self.B
        self.A = A_prev + self.dsr_eta * (ret - A_prev)
        self.B = B_prev + self.dsr_eta * (ret * ret - B_prev)
        denominator = max((B_prev - A_prev * A_prev) ** 1.5, 1e-8)
        return float((B_prev * ret - 0.5 * A_prev * ret * ret) / denominator)

    def step(self, action: Union[np.ndarray, float]) -> Tuple[np.ndarray, float, bool, Dict[str, float]]:
        action_value = float(action[0]) if isinstance(action, np.ndarray) else float(action)
        action_value = float(np.clip(action_value, -1.0, 1.0))
        price = float(self.df['Close'].iloc[self.current_step])
        if self.regime_detector is not None:
            action_value *= self.regime_detector.get_position_multiplier()
        target_position_value = action_value * self.max_position * self.portfolio_value
        current_position_value = self.shares * price
        trade_value = target_position_value - current_position_value
        if abs(trade_value) > 10.0:
            shares_delta = trade_value / price
            trade_cost = abs(trade_value) * self.transaction_cost
            self.balance -= shares_delta * price + trade_cost
            self.shares += shares_delta
            self.trades.append(
                Trade(
                    timestamp=self.current_step,
                    action='buy' if shares_delta > 0 else 'sell',
                    shares=shares_delta,
                    price=price,
                    cost=trade_cost,
                )
            )
        prev_value = self.portfolio_value
        self.portfolio_value = self.balance + self.shares * price
        ret = (self.portfolio_value / prev_value - 1.0) if prev_value else 0.0
        self.returns_history.append(ret)
        reward = self._calculate_dsr_reward(ret)
        self.portfolio_history.append(self.portfolio_value)
        if self.regime_detector is not None:
            self.regime_detector.detect_regime(np.asarray([self.df['returns'].iloc[self.current_step]], dtype=np.float32), float(self.df['atr_14'].iloc[self.current_step]))
        self.current_step += 1
        done = self.current_step >= len(self.df) - 1
        info = {
            'portfolio_value': float(self.portfolio_value),
            'return': float(ret),
            'shares': float(self.shares),
            'balance': float(self.balance),
        }
        return self._get_observation(), float(reward), done, info

    def get_portfolio_metrics(self) -> Dict[str, float]:
        if len(self.portfolio_history) < 2:
            return {'total_return': 0.0, 'sharpe_ratio': 0.0, 'max_drawdown': 0.0, 'num_trades': 0}
        returns = np.asarray(self.returns_history, dtype=np.float32)
        total_return = float(self.portfolio_history[-1] / self.initial_balance - 1.0)
        sharpe = float((returns.mean() / returns.std()) * np.sqrt(252 * 390)) if returns.std() > 0 else 0.0
        portfolio = np.asarray(self.portfolio_history, dtype=np.float32)
        peak = np.maximum.accumulate(portfolio)
        drawdown = (portfolio - peak) / peak
        return {
            'total_return': total_return,
            'sharpe_ratio': sharpe,
            'max_drawdown': float(drawdown.min()),
            'num_trades': float(len(self.trades)),
            'final_value': float(self.portfolio_history[-1]),
        }


class ActorCritic(nn.Module):
    def __init__(self, state_dim: int, action_dim: int = 1, hidden_dim: int = 256):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.Tanh(),
        )
        self.actor_mean = nn.Linear(hidden_dim, action_dim)
        self.actor_log_std = nn.Parameter(torch.zeros(action_dim))
        self.critic = nn.Linear(hidden_dim, 1)

    def forward(self, state: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        features = self.shared(state)
        mean = torch.tanh(self.actor_mean(features))
        std = torch.exp(self.actor_log_std).expand_as(mean)
        value = self.critic(features)
        return mean, std, value

    def get_action(self, state: torch.Tensor, deterministic: bool = False) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        mean, std, value = self.forward(state)
        dist = Normal(mean, std)
        action = mean if deterministic else dist.sample()
        log_prob = dist.log_prob(action).sum(dim=-1)
        return action, log_prob, value


class PPO:
    def __init__(
        self,
        state_dim: int,
        action_dim: int = 1,
        lr: float = 3e-4,
        gamma: float = 0.99,
        gae_lambda: float = 0.95,
        clip_epsilon: float = 0.2,
        value_coef: float = 0.5,
        entropy_coef: float = 0.01,
        max_grad_norm: float = 0.5,
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu',
    ):
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.clip_epsilon = clip_epsilon
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm
        self.device = device
        self.policy = ActorCritic(state_dim, action_dim).to(device)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=lr)

    def compute_gae(self, rewards: List[float], values: List[float], dones: List[bool], next_value: float) -> Tuple[List[float], List[float]]:
        advantages: List[float] = []
        gae = 0.0
        values = values + [next_value]
        for t in reversed(range(len(rewards))):
            if dones[t]:
                delta = rewards[t] - values[t]
                gae = delta
            else:
                delta = rewards[t] + self.gamma * values[t + 1] - values[t]
                gae = delta + self.gamma * self.gae_lambda * gae
            advantages.insert(0, gae)
        returns = [adv + val for adv, val in zip(advantages, values[:-1])]
        return advantages, returns

    def update(
        self,
        states: torch.Tensor,
        actions: torch.Tensor,
        old_log_probs: torch.Tensor,
        advantages: torch.Tensor,
        returns: torch.Tensor,
        epochs: int = 4,
        batch_size: int = 64,
    ) -> Dict[str, float]:
        metrics = {'policy_loss': [], 'value_loss': [], 'entropy': [], 'approx_kl': []}
        n = len(states)
        for _ in range(epochs):
            indices = torch.randperm(n, device=states.device)
            for start in range(0, n, batch_size):
                batch_idx = indices[start : start + batch_size]
                batch_states = states[batch_idx]
                batch_actions = actions[batch_idx]
                batch_old_log_probs = old_log_probs[batch_idx]
                batch_advantages = advantages[batch_idx]
                batch_returns = returns[batch_idx]

                mean, std, values = self.policy(batch_states)
                dist = Normal(mean, std)
                log_probs = dist.log_prob(batch_actions).sum(dim=-1)
                ratio = torch.exp(log_probs - batch_old_log_probs)
                surr1 = ratio * batch_advantages
                surr2 = torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * batch_advantages
                policy_loss = -torch.min(surr1, surr2).mean()
                value_loss = 0.5 * (values.squeeze(-1) - batch_returns).pow(2).mean()
                entropy = dist.entropy().mean()
                loss = policy_loss + self.value_coef * value_loss - self.entropy_coef * entropy
                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm)
                self.optimizer.step()
                with torch.no_grad():
                    approx_kl = ((ratio - 1.0) - ratio.log()).mean()
                metrics['policy_loss'].append(float(policy_loss.item()))
                metrics['value_loss'].append(float(value_loss.item()))
                metrics['entropy'].append(float(entropy.item()))
                metrics['approx_kl'].append(float(approx_kl.item()))
        return {key: float(np.mean(values)) for key, values in metrics.items()}


def train_ppo(env: StockTradingEnv, ppo: PPO, total_timesteps: int = 5000, rollout_length: int = 512, update_epochs: int = 4) -> List[float]:
    state = env.reset()
    states_buf: List[np.ndarray] = []
    actions_buf: List[np.ndarray] = []
    rewards_buf: List[float] = []
    values_buf: List[float] = []
    log_probs_buf: List[float] = []
    dones_buf: List[bool] = []
    episode_values: List[float] = []

    for timestep in range(total_timesteps):
        state_tensor = torch.as_tensor(state, dtype=torch.float32, device=ppo.device).unsqueeze(0)
        with torch.no_grad():
            action, log_prob, value = ppo.policy.get_action(state_tensor)
        next_state, reward, done, info = env.step(action.cpu().numpy()[0])
        states_buf.append(state)
        actions_buf.append(action.cpu().numpy()[0])
        rewards_buf.append(float(reward))
        values_buf.append(float(value.item()))
        log_probs_buf.append(float(log_prob.item()))
        dones_buf.append(bool(done))
        state = next_state
        if done:
            episode_values.append(float(info['portfolio_value']))
            state = env.reset()

        if len(states_buf) >= rollout_length:
            with torch.no_grad():
                next_state_tensor = torch.as_tensor(state, dtype=torch.float32, device=ppo.device).unsqueeze(0)
                _, _, next_value = ppo.policy(next_state_tensor)
                next_value_scalar = float(next_value.item())
            advantages, returns = ppo.compute_gae(rewards_buf, values_buf, dones_buf, next_value_scalar)
            states_tensor = torch.as_tensor(np.asarray(states_buf), dtype=torch.float32, device=ppo.device)
            actions_tensor = torch.as_tensor(np.asarray(actions_buf), dtype=torch.float32, device=ppo.device)
            old_log_probs_tensor = torch.as_tensor(np.asarray(log_probs_buf), dtype=torch.float32, device=ppo.device)
            advantages_tensor = torch.as_tensor(np.asarray(advantages), dtype=torch.float32, device=ppo.device)
            returns_tensor = torch.as_tensor(np.asarray(returns), dtype=torch.float32, device=ppo.device)
            advantages_tensor = (advantages_tensor - advantages_tensor.mean()) / (advantages_tensor.std() + 1e-8)
            metrics = ppo.update(
                states_tensor,
                actions_tensor,
                old_log_probs_tensor,
                advantages_tensor,
                returns_tensor,
                epochs=update_epochs,
                batch_size=RL_CONFIG['ppo_batch_size'],
            )
            print(f'PPO update @ step {timestep}: {metrics}')
            states_buf.clear()
            actions_buf.clear()
            rewards_buf.clear()
            values_buf.clear()
            log_probs_buf.clear()
            dones_buf.clear()
    return episode_values


selected_positions = rolling_backtester.selected_anchor_positions
if selected_positions is None or len(selected_positions) != len(rolling_logs):
    raise RuntimeError('Rolling logs are required before the RL section can run.')

rl_price_df = price_df.iloc[selected_positions][RAW_COLS].copy()
rl_market_df = build_rl_market_frame(rl_price_df)
rl_predictions = pad_prediction_paths(rolling_logs, horizon=HORIZON)

print({
    'rl_rows': len(rl_market_df),
    'prediction_tensor_shape': tuple(rl_predictions.shape),
    'run_rl_training': RUN_RL_TRAINING,
})

rl_env = StockTradingEnv(
    df=rl_market_df,
    predictions=rl_predictions,
    initial_balance=RL_CONFIG['initial_balance'],
    transaction_cost=RL_CONFIG['transaction_cost'],
    max_position=RL_CONFIG['max_position'],
    lookback=min(DEFAULT_LOOKBACK, len(rl_market_df) - 2),
    use_turbulence=True,
    dsr_eta=RL_CONFIG['dsr_eta'],
)

initial_state = rl_env.reset()
print('RL environment state dimension:', initial_state.shape[0])

probe_state, probe_reward, probe_done, probe_info = rl_env.step(np.asarray([0.1], dtype=np.float32))
print({
    'probe_reward': probe_reward,
    'probe_done': probe_done,
    'probe_portfolio_value': probe_info['portfolio_value'],
})

if RUN_RL_TRAINING:
    ppo = PPO(
        state_dim=rl_env.state_dim,
        action_dim=1,
        lr=RL_CONFIG['ppo_lr'],
        gamma=RL_CONFIG['ppo_gamma'],
        gae_lambda=RL_CONFIG['ppo_gae_lambda'],
        clip_epsilon=RL_CONFIG['ppo_clip_epsilon'],
        value_coef=RL_CONFIG['ppo_value_coef'],
        entropy_coef=RL_CONFIG['ppo_entropy_coef'],
        max_grad_norm=RL_CONFIG['ppo_max_grad_norm'],
        device=str(DEVICE),
    )
    episode_values = train_ppo(
        env=rl_env,
        ppo=ppo,
        total_timesteps=RL_CONFIG['rl_training_steps'],
        rollout_length=RL_CONFIG['ppo_rollout_length'],
        update_epochs=RL_CONFIG['ppo_update_epochs'],
    )
    print('RL training finished.')
    print('Episode portfolio values:', episode_values[-5:])
    print('Final portfolio metrics:', rl_env.get_portfolio_metrics())
else:
    print('Set RUN_RL_TRAINING = True to launch PPO training once data access and runtime are available.')


### Validation checklist status
- Strict causality per frame (`assert context.index[-1] < prediction_time`): **asserted**
- First predicted candle timestamp equals anchor timestamp: **asserted**
- Prediction count equals expected minute anchors: **asserted**
- No fan overlays used: **replaced by standalone per-frame PNG rendering**
- Frames saved sequentially to `frames/frame_0000.png ...`: **implemented**